# Federated unlearning for autoencoder anomaly detection: experiments as run

This is the working notebook from the experiments reported in *Utility Is Not
Compliance: Verifying Federated Unlearning for Autoencoder-Based Anomaly
Detection*

1. environment, data preparation and verification (sections 1 to 8)
2. the six runs behind Tables 1 and 3, each printing its own results table
3. one superseded run, kept and labelled, which reused the previous seed's
   base federation
4. the FedEraser rescue attempts the paper refers to
5. the push-weight ablation, which is exploratory and not in the paper
6. the determinism investigation, which is a negative result the paper's limitations section relies on



## 1. Check GPU

In [ ]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

Fri Aug  7 11:31:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   46C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Paths and parameters

In [ ]:
from pathlib import Path

# --- Drive locations ---
PROJECT_DIR   = "/content/drive/MyDrive/fed-unlearn-project"
CODE_ZIP      = f"{PROJECT_DIR}/fed-unlearn-ae-rewritten.zip"
RAW_DATA_ZIP  = f"{PROJECT_DIR}/MICCAI_FeTS2022_TrainingData.zip"
PARTITION_CSV = f"{PROJECT_DIR}/partitioning_1.csv"

# Backups written to Drive
PROCESSED_ZIP_DRIVE = f"{PROJECT_DIR}/processed_p1.zip"
OUTPUTS_DRIVE       = f"{PROJECT_DIR}/outputs_rewrite"

# --- Local scratch on the Colab VM (fast) ---
CODE_LOCAL      = "/content/fed-unlearn-ae"
PROCESSED_LOCAL = "/content/processed"
OUTPUTS_LOCAL   = "/content/outputs"

# Preprocessing controls (fixed across all experiment seeds).
PREPROCESS_SEED  = 42       # controls ONLY the train/val/test case split
MODALITY         = "flair"
SLICE_SIZE       = 128
N_CLIENTS        = 6
MIN_BRAIN_FRAC   = 0.20

Path(OUTPUTS_LOCAL).mkdir(parents=True, exist_ok=True)
print("Paths set.")

Paths set.


## 4. Stage the code

In [ ]:
import shutil, zipfile, sys

for p in ["/content/_code_tmp", CODE_LOCAL]:
    if Path(p).exists():
        shutil.rmtree(p)

assert Path(CODE_ZIP).exists(), f"Code zip not found at {CODE_ZIP}"
with zipfile.ZipFile(CODE_ZIP) as z:
    z.extractall("/content/_code_tmp")

# Package root(the directory that directly contains configs/base.yaml)
marker = next(Path("/content/_code_tmp").rglob("configs/base.yaml"), None)
assert marker is not None, "configs/base.yaml not found inside the zip"
pkg_root = marker.parent.parent
shutil.move(str(pkg_root), CODE_LOCAL)
shutil.rmtree("/content/_code_tmp", ignore_errors=True)

sys.path.insert(0, CODE_LOCAL)
print("CODE_LOCAL =", CODE_LOCAL)
print(sorted([p.name for p in Path(CODE_LOCAL).iterdir()]))

CODE_LOCAL = /content/fed-unlearn-ae
['CHANGELOG_corrections.md', 'LICENSE', 'PROJECT_GUIDE.md', 'README.md', 'configs', 'data', 'evaluation', 'federated', 'models', 'notebooks', 'paper', 'requirements.txt', 'scripts', 'tests', 'unlearning']


## 5. Install dependencies

Installs Flower + Ray + the NIfTI readers preprocessing needs. Then checks
PyTorch still sees CUDA. If CUDA is
lost, it reinstalls a CUDA build.

In [ ]:
import subprocess, sys
# Top-level install (runs in Colab). flwr[simulation] + ray + NIfTI readers.
!pip install -q "flwr[simulation]>=1.7.0" "ray[default]>=2.5.0" nibabel SimpleITK 2>/dev/null

import torch
if not torch.cuda.is_available():
    print("CUDA NOT visible after install — reinstalling a CUDA torch build...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "--force-reinstall", "torch", "torchvision",
                    "--index-url", "https://download.pytorch.org/whl/cu121"], check=False)
    print("\n*** Runtime -> Restart session, then re-run sections 1-5. ***")
else:
    print("CUDA OK:", torch.cuda.get_device_name(0))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 123.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.3/129.3 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 112.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 104.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.5/310.

## 6. Self-test the methods

Runs the bundled `tests/test_methods.py`. All should PASS: FedAvg
weighting, FedEraser calibration (rescales to probed magnitude; excludes
BatchNorm stats), calibration direction preservation, KD clamp binding, and
seed determinism.

In [ ]:
import subprocess
r = subprocess.run(["python","-m","tests.test_methods"],
                   cwd=CODE_LOCAL, capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print("STDERR:\n", r.stderr)
    raise RuntimeError("Self-tests failed — stop and investigate before training.")

PASS test_fedavg_weighting
PASS test_calibration_rescales
PASS test_calibration_preserves_direction
PASS test_calibration_clips_scale
PASS test_calibration_zeroes_buffers
PASS test_calibration_excludes_bn_stats
PASS test_kd_clamp_binds
PASS test_seed_determinism
PASS test_memorisation_signal
PASS test_signal_to_scale
PASS test_derive_seed_is_process_stable
PASS test_seeded_shuffle_is_reproducible

12/12 tests passed.



### 7a. Unzip the raw FeTS data

In [ ]:
if DID_PREPROCESS:
    Path("/content/raw").mkdir(exist_ok=True)
    t0 = time.time()
    print("Unzipping 13 GB raw FeTS data — this takes a while...")
    with zipfile.ZipFile(RAW_DATA_ZIP) as z:
        z.extractall("/content/raw")
    print(f"Unzip done in {(time.time()-t0)/60:.1f} min")

    # Locate the data root: the folder directly containing the FeTS2022_* cases
    first_case = next((p for p in Path("/content/raw").rglob("FeTS2022_*")
                       if p.is_dir()), None)
    assert first_case is not None, "No FeTS2022_* case folders found after unzip"
    RAW_ROOT = str(first_case.parent)
    print("Detected RAW_ROOT:", RAW_ROOT)
    print("Sample case contents:")
    print(sorted([p.name for p in first_case.iterdir()])[:8])
else:
    print("Skipped (restored from backup).")

Unzipping 13 GB raw FeTS data — this takes a while...
Unzip done in 6.5 min
Detected RAW_ROOT: /content/raw/MICCAI_FeTS2022_TrainingData
Sample case contents:
['FeTS2022_00596_flair.nii.gz', 'FeTS2022_00596_seg.nii.gz', 'FeTS2022_00596_t1.nii.gz', 'FeTS2022_00596_t1ce.nii.gz', 'FeTS2022_00596_t2.nii.gz']


### 7b. Preprocess

Slices the FLAIR volumes to 128×128, tags healthy vs tumour, and splits
case-wise into train/val/test using the fixed seed.

In [ ]:
if DID_PREPROCESS:
    t0 = time.time()
    run_stream([
        "python", "data/preprocess_fets.py",
        "--raw_dir", RAW_ROOT,
        "--partition_csv", PARTITION_CSV,
        "--out_dir", PROCESSED_LOCAL,
        "--modality", MODALITY,
        "--slice_size", str(SLICE_SIZE),
        "--n_clients", str(N_CLIENTS),
        "--min_brain_fraction", str(MIN_BRAIN_FRAC),
        "--seed", str(PREPROCESS_SEED),
    ])
    print(f"\nPreprocessing took {(time.time()-t0)/60:.1f} min")
else:
    print("Skipped (restored from backup).")

$ python data/preprocess_fets.py --raw_dir /content/raw/MICCAI_FeTS2022_TrainingData --partition_csv /content/drive/MyDrive/fed-unlearn-project/partitioning_1.csv --out_dir /content/processed --modality flair --slice_size 128 --n_clients 6 --min_brain_fraction 0.2 --seed 42
Reading partitioning from /content/drive/MyDrive/fed-unlearn-project/partitioning_1.csv
  1251 cases in partitioning CSV
Selected top 6 partitions by case count:
  Client 0 <- Partition 1: 511 cases
  Client 1 <- Partition 18: 382 cases
  Client 2 <- Partition 4: 47 cases
  Client 3 <- Partition 13: 35 cases
  Client 4 <- Partition 21: 35 cases
  Client 5 <- Partition 6: 34 cases

Skipped 207 cases (not in selected partitions)
Skipped 0 cases (folder not found in /content/raw/MICCAI_FeTS2022_TrainingData)
Total cases retained: 1044

Client 0 / train:   0%|          | 0/357 [00:00<?, ?it/s]/content/fed-unlearn-ae/data/preprocess_fets.py:151: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pi

### 7c. Verify slice counts

In [ ]:
from data.datasets import BratsSliceDataset

print(f"{'client':<10}{'train healthy':>15}{'train tumour':>15}")
tot_h = tot_t = 0
for cid in range(N_CLIENTS):
    c = Path(PROCESSED_LOCAL) / f"client_{cid}"
    h = len(BratsSliceDataset(c, split="train", mode="healthy"))
    t = len(BratsSliceDataset(c, split="train", mode="tumour"))
    tot_h += h; tot_t += t
    print(f"client_{cid:<3}{h:>15}{t:>15}")
print(f"{'TOTAL':<10}{tot_h:>15}{tot_t:>15}")

s, m = BratsSliceDataset(Path(PROCESSED_LOCAL)/"client_0", "train", "healthy")[0]
print("\nslice shape:", tuple(s.shape), "dtype:", s.dtype,
      "| mask shape:", tuple(m.shape))
assert tuple(s.shape) == (1, SLICE_SIZE, SLICE_SIZE), "slice shape mismatch"
print("OK: data loads through BratsSliceDataset")

client      train healthy   train tumour
client_0             7057          18249
client_1             5250          14160
client_2              365           1669
client_3              388           1254
client_4              345           1419
client_5              378           1249
TOTAL               13783          38000

slice shape: (1, 128, 128) dtype: torch.float32 | mask shape: (1, 128, 128)
OK: data loads through BratsSliceDataset


### 7d. Back up the preprocessed data to Drive

Zips the processed slices and copies a single archive to Drive.

In [ ]:
if DID_PREPROCESS:
    print("Zipping processed data and copying to Drive (one-time backup)...")
    t0 = time.time()
    shutil.make_archive("/content/processed_p1", "zip", PROCESSED_LOCAL)
    shutil.copy2("/content/processed_p1.zip", PROCESSED_ZIP_DRIVE)
    print(f"Backed up to {PROCESSED_ZIP_DRIVE} in {(time.time()-t0)/60:.1f} min")
else:
    print("Skipped (already had a backup).")

Zipping processed data and copying to Drive (one-time backup)...
Backed up to /content/drive/MyDrive/fed-unlearn-project/processed_p1.zip in 0.8 min


## 8. Helper functions

Builds a per-run config (with the right experiment seed / forgotten client /
paths) and a helper to back up a run's outputs to Drive as a zip.

In [ ]:
import yaml, shutil, subprocess, sys

BASE_CONFIG = f"{CODE_LOCAL}/configs/base.yaml"

def run_stream(args):
    """Run a command from CODE_LOCAL, streaming stdout live. Raises on failure."""
    print("$", " ".join(args))
    proc = subprocess.Popen(args, cwd=CODE_LOCAL, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
        sys.stdout.flush()
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed ({proc.returncode}): {' '.join(args)}")

def make_run_config(seed, forgotten_client):
    with open(BASE_CONFIG) as f:
        cfg = yaml.safe_load(f)
    cfg["experiment"]["seed"] = seed
    cfg["experiment"]["output_dir"] = OUTPUTS_LOCAL
    cfg["data"]["processed_dir"] = PROCESSED_LOCAL
    cfg["unlearning"]["forgotten_client"] = forgotten_client
    out = f"/content/config_seed{seed}_forget{forgotten_client}.yaml"
    with open(out, "w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    return out

def backup_outputs(tag):
    '''Zip the current outputs and copy to Drive under OUTPUTS_DRIVE.'''
    Path(OUTPUTS_DRIVE).mkdir(parents=True, exist_ok=True)
    arch = f"/content/outputs_{tag}"
    shutil.make_archive(arch, "zip", OUTPUTS_LOCAL)
    dst = f"{OUTPUTS_DRIVE}/outputs_{tag}.zip"
    shutil.copy2(arch + ".zip", dst)
    print("Backed up outputs to", dst)

print("Helpers ready.")

Helpers ready.


---
## The runs behind the paper

The six cells below produced every number in Tables 1 and 3. Each prints its
own results table, so the figures in the paper can be traced to a specific
execution:

| seed | forgotten client | cell |
|---|---|---|
| 42 | 0 | the run in this section |
| 123 | 0 | later, the seed-isolated run |
| 2024 | 0 | later, the seed-isolated run |
| 42 | 2 | later |
| 123 | 2 | later |
| 2024 | 2 | later |

## 9. Parameters — set the experiment to run

**Start with the canary: `SEED=42, FORGOTTEN_CLIENT=0`**

In [ ]:
SEED = 42
FORGOTTEN_CLIENT = 0

RUN_CFG = make_run_config(SEED, FORGOTTEN_CLIENT)
RUN_TAG = f"seed{SEED}_forget{FORGOTTEN_CLIENT}"
print(f"Configured: seed={SEED}, forgotten_client={FORGOTTEN_CLIENT}")
print("Config:", RUN_CFG)

Configured: seed=42, forgotten_client=0
Config: /content/config_seed42_forget0.yaml


## 10. Base federation

Trains the pre-unlearning model on all clients (100 rounds).

In [ ]:
# Base federation depends on the seed only (not the forgotten client),
# so it is reused across forgotten-client choices for the same seed.
base_local = f"{OUTPUTS_LOCAL}/base_federation"
if Path(f"{base_local}/round_0100/global_model.pt").exists():
    print("Base federation already present for this session — skipping.")
else:
    run_stream(["python", "scripts/run_base_federation.py", "--config", RUN_CFG])

assert Path(f"{base_local}/round_0100/global_model.pt").exists(), "base model missing"
print("Base federation ready.")

$ python scripts/run_base_federation.py --config /content/config_seed42_forget0.yaml
2026-06-19 05:21:56.633979: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-19 05:21:57.766471: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a new Flower app from a template

		$ flwr run  # Run the Flower app in Simulation Mode

	Using `start_simulation()` is deprecated.

            Thi

## 11. Oracle (retrain without the forgotten client)

Also 100 rounds, on the remaining clients only. The reference every method is
measured against.

In [ ]:
oracle_local = f"{OUTPUTS_LOCAL}/oracle_forgot_{FORGOTTEN_CLIENT}"
if Path(f"{oracle_local}/round_0100/global_model.pt").exists():
    print("Oracle already present for this (session, client) — skipping.")
else:
    run_stream(["python", "scripts/run_oracle.py", "--config", RUN_CFG,
                "--forgotten_client", str(FORGOTTEN_CLIENT)])

assert Path(f"{oracle_local}/round_0100/global_model.pt").exists(), "oracle missing"
print("Oracle ready.")

$ python scripts/run_oracle.py --config /content/config_seed42_forget0.yaml --forgotten_client 0
2026-06-19 06:00:35.646458: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-19 06:00:35.720206: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a new Flower app from a template

		$ flwr run  # Run the Flower app in Simulation Mode

	Using `start_simulation()` is deprecated.

   

## 12. Unlearning — gradient ascent, federated KD, FedEraser


In [ ]:
run_stream([
    "python", "scripts/run_unlearning.py",
    "--config", RUN_CFG,
    "--forgotten_client", str(FORGOTTEN_CLIENT),
    "--base_run_dir", f"{OUTPUTS_LOCAL}/base_federation",
])

$ python scripts/run_unlearning.py --config /content/config_seed42_forget0.yaml --forgotten_client 0 --base_run_dir /content/outputs/base_federation

UNLEARNING 1/3: Gradient Ascent (Halimi et al. 2022)

[grad_ascent] Loading base model from /content/outputs/base_federation/round_0100/global_model.pt

[grad_ascent] Phase 1: gradient ascent on client 0
  steps=2, lr=1e-05
2026-06-19 08:02:15.504045: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-19 08:02:15.577356: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


### Record of the seed 42, client 0 run

The output below is the only surviving record of this run. Its archive,
`outputs_seed42_forget0.zip`, was written at the end of this cell and later
found to be corrupt, so it was deleted. Every value in the paper's Table 3 row
for seed 42, client 0 is the output printed here.

This run trained its own base federation into `/content/outputs`. The later
seed-isolated runs wrote to `/content/outputs_seed{SEED}` instead, so seed 42
has two base federations: the one below, used for the client 0 regime, and a
second one used for the client 2 regime. Unseeded shuffling makes them
different models. Re-evaluating client 0 against the second base gives a
pre-unlearning AUROC of 0.858 rather than 0.895.

## 13. Evaluation + results

Computes anomaly-detection AUROC/Dice, Wasserstein/KS efficacy vs oracle, and
the MIA, then prints the headline table and backs everything up to Drive.

In [ ]:
manifest = f"{OUTPUTS_LOCAL}/manifest_forgot_{FORGOTTEN_CLIENT}.json"
run_stream([
    "python", "scripts/run_evaluation.py",
    "--config", RUN_CFG,
    "--forgotten_client", str(FORGOTTEN_CLIENT),
    "--manifest", manifest,
])

import json
res = json.load(open(f"{OUTPUTS_LOCAL}/evaluation_results_forgot_{FORGOTTEN_CLIENT}.json"))
print(f"\n=== seed {SEED}, forgotten client {FORGOTTEN_CLIENT} ===")
print(f"{'method':<16}{'AUROC_rem':>11}{'Dice_rem':>10}{'W_oracle':>11}{'MIA':>9}")
for name in ["pre_unlearning","oracle","grad_ascent","kd","fed_eraser"]:
    m = res["metrics"].get(name, {})
    ad = m.get("anomaly_detection", {}).get("remaining", {}).get("test", {})
    auroc = ad.get("auroc", float("nan"))
    dice  = ad.get("dice", float("nan"))
    w = 0.0 if name == "oracle" else m.get("efficacy", {}).get("wasserstein", float("nan"))
    mia = m.get("mia", {}).get("attack_auroc", float("nan"))
    print(f"{name:<16}{auroc:>11.4f}{dice:>10.4f}{w:>11.4f}{mia:>9.4f}")

backup_outputs(RUN_TAG)

$ python scripts/run_evaluation.py --config /content/config_seed42_forget0.yaml --forgotten_client 0 --manifest /content/outputs/manifest_forgot_0.json

EVALUATION 1/3: Anomaly Detection (pixel AUROC, Dice)

  pre_unlearning: /content/outputs/base_federation/round_0100/global_model.pt
    remaining: AUROC=0.8951, Dice=0.2697
    forgotten: AUROC=0.9222, Dice=0.3639

  oracle: /content/outputs/oracle_forgot_0/round_0100/global_model.pt
    remaining: AUROC=0.8463, Dice=0.1602
    forgotten: AUROC=0.8699, Dice=0.1926

  grad_ascent: /content/outputs/unlearn_grad_ascent_forgot_0/final_unlearned_model.pt
    remaining: AUROC=0.8627, Dice=0.1890
    forgotten: AUROC=0.8900, Dice=0.2561

  kd: /content/outputs/unlearn_kd_forgot_0/final_unlearned_model.pt
    remaining: AUROC=0.9526, Dice=0.5051
    forgotten: AUROC=0.9768, Dice=0.6736

  fed_eraser: /content/outputs/unlearn_fed_eraser_forgot_0/final_unlearned_model.pt
    remaining: AUROC=0.8370, Dice=0.1538
    forgotten: AUROC=0.8638, Dice

## 14. Running the remaining combinations

In [ ]:
import json
log = json.load(open("/content/outputs/unlearn_kd_forgot_0/distillation_log.json"))
print("KD distillation rounds:", len(log))
for r in log[:3] + log[-3:]:
    print(r)

KD distillation rounds: 20
{'round': 1, 'phase': 'fit', 'metrics': {'client_id': 1.5483199524234315, 'match': 0.03250633432425171, 'push_dist': 0.020403234448369393}, 'client_metrics': [{'client_id': 5, 'match': 0.04142670126801187, 'push_dist': 0.02237674861469052}, {'client_id': 3, 'match': 0.035444216709583996, 'push_dist': 0.019399375026114284}, {'client_id': 2, 'match': 0.035957731983878395, 'push_dist': 0.018929532475092194}, {'client_id': 4, 'match': 0.030166857056319715, 'push_dist': 0.018684989716857673}, {'client_id': 1, 'match': 0.03156072717023695, 'push_dist': 0.020550702020480503}]}
{'round': 2, 'phase': 'fit', 'metrics': {'client_id': 1.5483199524234315, 'match': 0.03246612615484567, 'push_dist': 0.02164345085988589}, 'client_metrics': [{'client_id': 3, 'match': 0.034810845057169595, 'push_dist': 0.02035760097205639}, {'client_id': 5, 'match': 0.039643163267861714, 'push_dist': 0.022282394224947148}, {'client_id': 1, 'match': 0.0316370773969627, 'push_dist': 0.0220031096

In [ ]:
import json, os
fe_dir = "/content/outputs/unlearn_fed_eraser_forgot_0"
print(os.listdir(fe_dir))
# if a calibration log or training_log exists, peek at it
for fn in ["calibration/training_log.json", "training_log.json"]:
    p = os.path.join(fe_dir, fn)
    if os.path.exists(p):
        log = json.load(open(p))
        print(f"\n{fn}: {len(log)} entries")
        for r in (log[:2] + log[-2:]):
            print({k: r.get(k) for k in ("round","phase","loss","metrics")})

['post_replay_model.pt', 'final_unlearned_model.pt', 'calibration']

calibration/training_log.json: 40 entries
{'round': 1, 'phase': 'fit', 'loss': None, 'metrics': {'train_loss': 0.08599091419193344, 'client_id': 1.5483199524234312}}
{'round': 1, 'phase': 'evaluate', 'loss': 0.07366596393427563, 'metrics': {'val_loss': 0.07366596393427563, 'client_id': 1.562456385205862}}
{'round': 20, 'phase': 'fit', 'loss': None, 'metrics': {'train_loss': 0.030587479548302515, 'client_id': 1.5483199524234312}}
{'round': 20, 'phase': 'evaluate', 'loss': 0.030054632804388363, 'metrics': {'val_loss': 0.03005463280438836, 'client_id': 1.562456385205862}}


In [ ]:
import torch, sys
sys.path.insert(0, CODE_LOCAL)
from models.ae_2d import AE2D, reconstruction_loss
from data.datasets import get_client_loader
from federated.client import set_parameters
from pathlib import Path
import yaml

cfg = yaml.safe_load(open(RUN_CFG))
m = cfg["model"]
dev = "cuda" if torch.cuda.is_available() else "cpu"

def recon_loss_of(ckpt_path, client_id=1, n_batches=10):
    ck = torch.load(ckpt_path, weights_only=False)
    net = AE2D(m["in_channels"], m["base_channels"], m["latent_channels"]).to(dev)
    set_parameters(net, [p.numpy() for p in ck["parameters"]])
    net.eval()
    dl = get_client_loader(Path(PROCESSED_LOCAL)/f"client_{client_id}",
                           split="val", mode="healthy", batch_size=32, shuffle=False)
    tot, n = 0.0, 0
    with torch.no_grad():
        for i,(x,_) in enumerate(dl):
            if i>=n_batches: break
            x=x.to(dev); tot += reconstruction_loss(net(x), x, m["loss"]).item(); n+=1
    return tot/max(n,1)

base = "/content/outputs/base_federation/round_0100/global_model.pt"
post = "/content/outputs/unlearn_fed_eraser_forgot_0/post_replay_model.pt"
g1   = "/content/outputs/base_federation/round_0001/global_model.pt"
final= "/content/outputs/unlearn_fed_eraser_forgot_0/final_unlearned_model.pt"

print("base (round 100) val recon:", round(recon_loss_of(base),4))
print("G_1  (round 1)   val recon:", round(recon_loss_of(g1),4))
print("post-replay      val recon:", round(recon_loss_of(post),4))
print("fed_eraser final val recon:", round(recon_loss_of(final),4))

base (round 100) val recon: 0.0169
G_1  (round 1)   val recon: 0.1245
post-replay      val recon: 0.1152
fed_eraser final val recon: 0.0343


In [ ]:
import yaml
cfg = yaml.safe_load(open(RUN_CFG))
cfg["unlearning"]["fed_eraser"]["replay_stride"] = 10
yaml.safe_dump(cfg, open(RUN_CFG, "w"), sort_keys=False)
print("replay_stride set to 10")

run_stream(["python", "scripts/run_unlearning.py",
            "--config", RUN_CFG, "--forgotten_client", "0",
            "--base_run_dir", f"{OUTPUTS_LOCAL}/base_federation",
            "--methods", "fed_eraser"])

replay_stride set to 10
$ python scripts/run_unlearning.py --config /content/config_seed42_forget0.yaml --forgotten_client 0 --base_run_dir /content/outputs/base_federation --methods fed_eraser

UNLEARNING 3/3: FedEraser (Liu et al. 2021)

[fed_eraser] Calibrated replay without client 0
  Remaining clients: [1, 2, 3, 4, 5]
  Rounds: 100, stride: 10, probe steps/client/round: 1
2026-06-19 08:46:22.649853: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-19 08:46:22.721901: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler f

In [ ]:
import yaml
cfg = yaml.safe_load(open(RUN_CFG))
cfg["unlearning"]["fed_eraser"]["replay_stride"] = 1
yaml.safe_dump(cfg, open(RUN_CFG, "w"), sort_keys=False)
print("replay_stride reset to 1")

replay_stride reset to 1


---
### Superseded run, do not read the numbers below as seed 123

This cell is labelled `SEED = 123` but it reused the base federation already on
disk from the seed 42 run, because the guard only checked whether a base model
existed and not which seed produced it. Its output therefore repeats the seed
42 values: pre-unlearning 0.8951, oracle 0.8463.

The next cell confirms the reuse. The run was discarded. The seed 123 numbers utilise come from the seed-isolated cell further down, which writes to
`/content/outputs_seed123` and reports pre-unlearning 0.8608.

In [ ]:
# FULL RUN for one (seed, forgotten_client) combination
SEED = 123
FORGOTTEN_CLIENT = 0

import sys, subprocess, json, yaml, time
from pathlib import Path

# --- 0. Prerequisite checks (fail loudly, not mid-run) ---
for name in ["CODE_LOCAL", "PROCESSED_LOCAL", "OUTPUTS_LOCAL", "BASE_CONFIG"]:
    assert name in dir(), f"{name} not defined — run sections 3 and 8 first."
assert Path(f"{PROCESSED_LOCAL}/client_0/train").exists(), \
    "Processed data missing — run section 7 first."
assert Path(f"{CODE_LOCAL}/scripts/run_base_federation.py").exists(), \
    "Code not staged — run section 4 first."

# --- 1. Ensure run_stream exists (re-define defensively) ---
def run_stream(args):
    print("$", " ".join(args)); sys.stdout.flush()
    proc = subprocess.Popen(args, cwd=CODE_LOCAL, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end=""); sys.stdout.flush()
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"FAILED ({proc.returncode}): {' '.join(args)}")

# --- 2. Build per-run config, with replay_stride forced back to 1 ---
with open(BASE_CONFIG) as f:
    cfg = yaml.safe_load(f)
cfg["experiment"]["seed"] = SEED
cfg["experiment"]["output_dir"] = OUTPUTS_LOCAL
cfg["data"]["processed_dir"] = PROCESSED_LOCAL
cfg["unlearning"]["forgotten_client"] = FORGOTTEN_CLIENT
cfg["unlearning"]["fed_eraser"]["replay_stride"] = 1   # undo the stride-10/1000 experiments
RUN_CFG = f"/content/config_seed{SEED}_forget{FORGOTTEN_CLIENT}.yaml"
with open(RUN_CFG, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
RUN_TAG = f"seed{SEED}_forget{FORGOTTEN_CLIENT}"
print(f"Config: seed={SEED}, client={FORGOTTEN_CLIENT}, replay_stride=1\n")

t_start = time.time()

# --- 3. Base federation (skip if already on disk this session) ---
base_local = f"{OUTPUTS_LOCAL}/base_federation"
if Path(f"{base_local}/round_0100/global_model.pt").exists():
    print(">>> Base federation already present — skipping.\n")
else:
    print(">>> Base federation (multi-hour stage)...\n")
    run_stream(["python", "scripts/run_base_federation.py", "--config", RUN_CFG])
assert Path(f"{base_local}/round_0100/global_model.pt").exists(), "base missing"

# --- 4. Oracle ---
oracle_local = f"{OUTPUTS_LOCAL}/oracle_forgot_{FORGOTTEN_CLIENT}"
if Path(f"{oracle_local}/round_0100/global_model.pt").exists():
    print(">>> Oracle already present — skipping.\n")
else:
    print(">>> Oracle...\n")
    run_stream(["python", "scripts/run_oracle.py", "--config", RUN_CFG,
                "--forgotten_client", str(FORGOTTEN_CLIENT)])
assert Path(f"{oracle_local}/round_0100/global_model.pt").exists(), "oracle missing"

# --- 5. Unlearning (all three methods) ---
print(">>> Unlearning (grad_ascent, kd, fed_eraser)...\n")
run_stream(["python", "scripts/run_unlearning.py", "--config", RUN_CFG,
            "--forgotten_client", str(FORGOTTEN_CLIENT),
            "--base_run_dir", base_local])

# --- 6. Evaluation ---
print(">>> Evaluation...\n")
manifest = f"{OUTPUTS_LOCAL}/manifest_forgot_{FORGOTTEN_CLIENT}.json"
run_stream(["python", "scripts/run_evaluation.py", "--config", RUN_CFG,
            "--forgotten_client", str(FORGOTTEN_CLIENT), "--manifest", manifest])

# --- 7. Results table ---
res = json.load(open(f"{OUTPUTS_LOCAL}/evaluation_results_forgot_{FORGOTTEN_CLIENT}.json"))
print(f"\n=== seed {SEED}, forgotten client {FORGOTTEN_CLIENT} ===")
print(f"{'method':<16}{'AUROC_rem':>11}{'Dice_rem':>10}{'W_oracle':>11}{'KS':>8}{'MIA':>9}")
for name in ["pre_unlearning","oracle","grad_ascent","kd","fed_eraser"]:
    m = res["metrics"].get(name, {})
    ad  = m.get("anomaly_detection", {}).get("remaining", {}).get("test", {})
    eff = m.get("efficacy", {})
    auroc = ad.get("auroc", float("nan"))
    dice  = ad.get("dice", float("nan"))
    w  = 0.0 if name == "oracle" else eff.get("wasserstein", float("nan"))
    ks = 0.0 if name == "oracle" else eff.get("ks_statistic", float("nan"))
    mia = m.get("mia", {}).get("attack_auroc", float("nan"))
    print(f"{name:<16}{auroc:>11.4f}{dice:>10.4f}{w:>11.4f}{ks:>8.4f}{mia:>9.4f}")

# --- 8. FedEraser sanity (post-replay should be finite, not NaN) ---
try:
    import torch
    from models.ae_2d import AE2D, reconstruction_loss
    from data.datasets import get_client_loader
    from federated.client import set_parameters
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    mm = cfg["model"]
    def recon_of(p, cid=1, nb=10):
        ck = torch.load(p, weights_only=False)
        net = AE2D(mm["in_channels"], mm["base_channels"], mm["latent_channels"]).to(dev)
        set_parameters(net, [t.numpy() for t in ck["parameters"]]); net.eval()
        dl = get_client_loader(Path(PROCESSED_LOCAL)/f"client_{cid}", "val", "healthy", 32, False)
        tot=n=0
        with torch.no_grad():
            for i,(x,_) in enumerate(dl):
                if i>=nb: break
                x=x.to(dev); tot+=reconstruction_loss(net(x),x,mm["loss"]).item(); n+=1
        return tot/max(n,1)
    pr = f"{OUTPUTS_LOCAL}/unlearn_fed_eraser_forgot_{FORGOTTEN_CLIENT}/post_replay_model.pt"
    if Path(pr).exists():
        print(f"\n[FedEraser sanity] post-replay val recon = {recon_of(pr):.4f}  "
              f"(NaN/blow-up = bad; we expect a finite number)")
except Exception as e:
    print("post-replay check skipped:", e)

# --- 9. Back up to Drive ---
import shutil
Path(OUTPUTS_DRIVE).mkdir(parents=True, exist_ok=True)
shutil.make_archive(f"/content/outputs_{RUN_TAG}", "zip", OUTPUTS_LOCAL)
shutil.copy2(f"/content/outputs_{RUN_TAG}.zip", f"{OUTPUTS_DRIVE}/outputs_{RUN_TAG}.zip")
print(f"\nBacked up to {OUTPUTS_DRIVE}/outputs_{RUN_TAG}.zip")
print(f"Total run time: {(time.time()-t_start)/60:.1f} min")

Config: seed=123, client=0, replay_stride=1

>>> Base federation already present — skipping.

>>> Oracle already present — skipping.

>>> Unlearning (grad_ascent, kd, fed_eraser)...

$ python scripts/run_unlearning.py --config /content/config_seed123_forget0.yaml --forgotten_client 0 --base_run_dir /content/outputs/base_federation

UNLEARNING 1/3: Gradient Ascent (Halimi et al. 2022)

[grad_ascent] Loading base model from /content/outputs/base_federation/round_0100/global_model.pt

[grad_ascent] Phase 1: gradient ascent on client 0
  steps=2, lr=1e-05
2026-06-19 09:30:38.830261: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-19 09:30:38.905004: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructio

In [ ]:
import os, time
base = "/content/outputs/base_federation/round_0100/global_model.pt"
if os.path.exists(base):
    print("old base modified:", time.ctime(os.path.getmtime(base)))
    print("(if this is from your seed-42 run, the seed-123 run reused it — confirmed)")
else:
    print("no old base federation found")

no old base federation found


### Seed-isolated runs

Each seed now writes to `/content/outputs_seed{SEED}`, so a later seed cannot
inherit an earlier one's base federation. The next four cells are the runs the
paper reports for seed 123 client 0, seed 2024 client 0, and both seeds for
client 2.

In [ ]:
# SEED-ISOLATED FULL RUN — one (seed, forgotten_client) combo.
# Each seed writes to /content/outputs_seed{SEED}
SEED = 123
FORGOTTEN_CLIENT = 0

import sys, subprocess, json, yaml, time, shutil
from pathlib import Path

# --- prerequisites (fail loudly, not mid-run) ---
for name in ["CODE_LOCAL", "PROCESSED_LOCAL", "BASE_CONFIG", "OUTPUTS_DRIVE"]:
    assert name in dir(), f"{name} not defined — run sections 3 and 8 first."
assert Path(f"{PROCESSED_LOCAL}/client_0/train").exists(), "Processed data missing (section 7)."
assert Path(f"{CODE_LOCAL}/scripts/run_base_federation.py").exists(), "Code not staged (section 4)."

# --- per-seed output dir (THE key fix: isolated, seed-tagged) ---
OUT = f"/content/outputs_seed{SEED}"
Path(OUT).mkdir(parents=True, exist_ok=True)
print(f"Output dir for this run: {OUT}")

# --- run_stream (defensive re-define; streams live output) ---
def run_stream(args):
    print("$", " ".join(args)); sys.stdout.flush()
    p = subprocess.Popen(args, cwd=CODE_LOCAL, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end=""); sys.stdout.flush()
    p.wait()
    if p.returncode != 0:
        raise RuntimeError(f"FAILED ({p.returncode}): {' '.join(args)}")

# --- per-run config: seed, this run's OUT, processed dir, stride reset ---
with open(BASE_CONFIG) as f:
    cfg = yaml.safe_load(f)
cfg["experiment"]["seed"] = SEED
cfg["experiment"]["output_dir"] = OUT
cfg["data"]["processed_dir"] = PROCESSED_LOCAL
cfg["unlearning"]["forgotten_client"] = FORGOTTEN_CLIENT
cfg["unlearning"]["fed_eraser"]["replay_stride"] = 1
RUN_CFG = f"/content/config_seed{SEED}_forget{FORGOTTEN_CLIENT}.yaml"
with open(RUN_CFG, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
RUN_TAG = f"seed{SEED}_forget{FORGOTTEN_CLIENT}"
print(f"Config: seed={SEED}, client={FORGOTTEN_CLIENT}\n")

t0 = time.time()

# --- 1. base federation (fresh, because OUT is empty for this seed) ---
base_local = f"{OUT}/base_federation"
if Path(f"{base_local}/round_0100/global_model.pt").exists():
    print(">>> base federation already done for this seed in this session — skipping.\n")
else:
    print(">>> base federation (this is the multi-HOUR stage; keep the tab alive)...\n")
    run_stream(["python", "scripts/run_base_federation.py", "--config", RUN_CFG])
assert Path(f"{base_local}/round_0100/global_model.pt").exists(), "base missing"

# --- 2. oracle ---
oracle_local = f"{OUT}/oracle_forgot_{FORGOTTEN_CLIENT}"
if Path(f"{oracle_local}/round_0100/global_model.pt").exists():
    print(">>> oracle already done — skipping.\n")
else:
    print(">>> oracle...\n")
    run_stream(["python", "scripts/run_oracle.py", "--config", RUN_CFG,
                "--forgotten_client", str(FORGOTTEN_CLIENT)])
assert Path(f"{oracle_local}/round_0100/global_model.pt").exists(), "oracle missing"

# --- 3. unlearning (all three) ---
print(">>> unlearning (grad_ascent, kd, fed_eraser)...\n")
run_stream(["python", "scripts/run_unlearning.py", "--config", RUN_CFG,
            "--forgotten_client", str(FORGOTTEN_CLIENT),
            "--base_run_dir", base_local])

# --- 4. evaluation ---
print(">>> evaluation...\n")
manifest = f"{OUT}/manifest_forgot_{FORGOTTEN_CLIENT}.json"
run_stream(["python", "scripts/run_evaluation.py", "--config", RUN_CFG,
            "--forgotten_client", str(FORGOTTEN_CLIENT), "--manifest", manifest])

# --- 5. results table ---
res = json.load(open(f"{OUT}/evaluation_results_forgot_{FORGOTTEN_CLIENT}.json"))
print(f"\n=== seed {SEED}, forgotten client {FORGOTTEN_CLIENT} ===")
print(f"{'method':<16}{'AUROC_rem':>11}{'Dice_rem':>10}{'W_oracle':>11}{'KS':>8}{'MIA':>9}")
for name in ["pre_unlearning","oracle","grad_ascent","kd","fed_eraser"]:
    m = res["metrics"].get(name, {})
    ad  = m.get("anomaly_detection", {}).get("remaining", {}).get("test", {})
    eff = m.get("efficacy", {})
    auroc = ad.get("auroc", float("nan")); dice = ad.get("dice", float("nan"))
    w  = 0.0 if name=="oracle" else eff.get("wasserstein", float("nan"))
    ks = 0.0 if name=="oracle" else eff.get("ks_statistic", float("nan"))
    mia = m.get("mia", {}).get("attack_auroc", float("nan"))
    print(f"{name:<16}{auroc:>11.4f}{dice:>10.4f}{w:>11.4f}{ks:>8.4f}{mia:>9.4f}")

# --- 6. FedEraser post-replay sanity (expected to be far off — confirms non-transfer) ---
try:
    import torch
    from models.ae_2d import AE2D, reconstruction_loss
    from data.datasets import get_client_loader
    from federated.client import set_parameters
    dev = "cuda" if torch.cuda.is_available() else "cpu"; mm = cfg["model"]
    def recon_of(p, cid=1, nb=10):
        ck = torch.load(p, weights_only=False)
        net = AE2D(mm["in_channels"], mm["base_channels"], mm["latent_channels"]).to(dev)
        set_parameters(net, [t.numpy() for t in ck["parameters"]]); net.eval()
        dl = get_client_loader(Path(PROCESSED_LOCAL)/f"client_{cid}", "val", "healthy", 32, False)
        tot=n=0
        with torch.no_grad():
            for i,(x,_) in enumerate(dl):
                if i>=nb: break
                x=x.to(dev); tot+=reconstruction_loss(net(x),x,mm["loss"]).item(); n+=1
        return tot/max(n,1)
    base_ck = f"{base_local}/round_0100/global_model.pt"
    pr = f"{OUT}/unlearn_fed_eraser_forgot_{FORGOTTEN_CLIENT}/post_replay_model.pt"
    print(f"\n[sanity] base(round100) recon = {recon_of(base_ck):.4f}")
    if Path(pr).exists():
        print(f"[sanity] FedEraser post-replay recon = {recon_of(pr):.4f}  "
              f"(far above base = replay doesn't transfer, as expected)")
except Exception as e:
    print("sanity check skipped:", e)

# --- 7. back up THIS seed's outputs to Drive ---
Path(OUTPUTS_DRIVE).mkdir(parents=True, exist_ok=True)
shutil.make_archive(f"/content/outputs_{RUN_TAG}", "zip", OUT)
shutil.copy2(f"/content/outputs_{RUN_TAG}.zip", f"{OUTPUTS_DRIVE}/outputs_{RUN_TAG}.zip")
print(f"\nBacked up to {OUTPUTS_DRIVE}/outputs_{RUN_TAG}.zip")
print(f"Total run time: {(time.time()-t0)/60:.1f} min")

Streaming output truncated to the last 5000 lines.
INFO :      	               (70, 0.780955702825253),
INFO :      	               (71, 0.7809557028252528),
INFO :      	               (72, 0.780955702825253),
INFO :      	               (73, 0.780955702825253),
INFO :      	               (74, 0.780955702825253),
INFO :      	               (75, 0.780955702825253),
INFO :      	               (76, 0.780955702825253),
INFO :      	               (77, 0.780955702825253),
INFO :      	               (78, 0.780955702825253),
INFO :      	               (79, 0.780955702825253),
INFO :      	               (80, 0.780955702825253),
INFO :      	               (81, 0.7809557028252528),
INFO :      	               (82, 0.7809557028252528),
INFO :      	               (83, 0.780955702825253),
INFO :      	               (84, 0.7809557028252528),
INFO :      	               (85, 0.7809557028252528),
INFO :      	               (86, 0.7809557028252528),
INFO :      	               (87, 0.7809557

In [ ]:
# SEED-ISOLATED FULL RUN — one (seed, forgotten_client) combo.
# Each seed writes to /content/outputs_seed{SEED}
SEED = 2024
FORGOTTEN_CLIENT = 0
import sys, subprocess, json, yaml, time, shutil
from pathlib import Path

# --- prerequisites (fail loudly, not mid-run) ---
for name in ["CODE_LOCAL", "PROCESSED_LOCAL", "BASE_CONFIG", "OUTPUTS_DRIVE"]:
    assert name in dir(), f"{name} not defined — run sections 3 and 8 first."
assert Path(f"{PROCESSED_LOCAL}/client_0/train").exists(), "Processed data missing (section 7)."
assert Path(f"{CODE_LOCAL}/scripts/run_base_federation.py").exists(), "Code not staged (section 4)."

# --- per-seed output dir (THE key fix: isolated, seed-tagged) ---
OUT = f"/content/outputs_seed{SEED}"
Path(OUT).mkdir(parents=True, exist_ok=True)
print(f"Output dir for this run: {OUT}")

# --- run_stream (defensive re-define; streams live output) ---
def run_stream(args):
    print("$", " ".join(args)); sys.stdout.flush()
    p = subprocess.Popen(args, cwd=CODE_LOCAL, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end=""); sys.stdout.flush()
    p.wait()
    if p.returncode != 0:
        raise RuntimeError(f"FAILED ({p.returncode}): {' '.join(args)}")

# --- per-run config: seed, this run's OUT, processed dir, stride reset ---
with open(BASE_CONFIG) as f:
    cfg = yaml.safe_load(f)
cfg["experiment"]["seed"] = SEED
cfg["experiment"]["output_dir"] = OUT
cfg["data"]["processed_dir"] = PROCESSED_LOCAL
cfg["unlearning"]["forgotten_client"] = FORGOTTEN_CLIENT
cfg["unlearning"]["fed_eraser"]["replay_stride"] = 1
RUN_CFG = f"/content/config_seed{SEED}_forget{FORGOTTEN_CLIENT}.yaml"
with open(RUN_CFG, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
RUN_TAG = f"seed{SEED}_forget{FORGOTTEN_CLIENT}"
print(f"Config: seed={SEED}, client={FORGOTTEN_CLIENT}\n")

t0 = time.time()

# --- 1. base federation (fresh, because OUT is empty for this seed) ---
base_local = f"{OUT}/base_federation"
if Path(f"{base_local}/round_0100/global_model.pt").exists():
    print(">>> base federation already done for this seed in this session — skipping.\n")
else:
    print(">>> base federation (this is the multi-HOUR stage; keep the tab alive)...\n")
    run_stream(["python", "scripts/run_base_federation.py", "--config", RUN_CFG])
assert Path(f"{base_local}/round_0100/global_model.pt").exists(), "base missing"

# --- 2. oracle ---
oracle_local = f"{OUT}/oracle_forgot_{FORGOTTEN_CLIENT}"
if Path(f"{oracle_local}/round_0100/global_model.pt").exists():
    print(">>> oracle already done — skipping.\n")
else:
    print(">>> oracle...\n")
    run_stream(["python", "scripts/run_oracle.py", "--config", RUN_CFG,
                "--forgotten_client", str(FORGOTTEN_CLIENT)])
assert Path(f"{oracle_local}/round_0100/global_model.pt").exists(), "oracle missing"

# --- 3. unlearning (all three) ---
print(">>> unlearning (grad_ascent, kd, fed_eraser)...\n")
run_stream(["python", "scripts/run_unlearning.py", "--config", RUN_CFG,
            "--forgotten_client", str(FORGOTTEN_CLIENT),
            "--base_run_dir", base_local])

# --- 4. evaluation ---
print(">>> evaluation...\n")
manifest = f"{OUT}/manifest_forgot_{FORGOTTEN_CLIENT}.json"
run_stream(["python", "scripts/run_evaluation.py", "--config", RUN_CFG,
            "--forgotten_client", str(FORGOTTEN_CLIENT), "--manifest", manifest])

# --- 5. results table ---
res = json.load(open(f"{OUT}/evaluation_results_forgot_{FORGOTTEN_CLIENT}.json"))
print(f"\n=== seed {SEED}, forgotten client {FORGOTTEN_CLIENT} ===")
print(f"{'method':<16}{'AUROC_rem':>11}{'Dice_rem':>10}{'W_oracle':>11}{'KS':>8}{'MIA':>9}")
for name in ["pre_unlearning","oracle","grad_ascent","kd","fed_eraser"]:
    m = res["metrics"].get(name, {})
    ad  = m.get("anomaly_detection", {}).get("remaining", {}).get("test", {})
    eff = m.get("efficacy", {})
    auroc = ad.get("auroc", float("nan")); dice = ad.get("dice", float("nan"))
    w  = 0.0 if name=="oracle" else eff.get("wasserstein", float("nan"))
    ks = 0.0 if name=="oracle" else eff.get("ks_statistic", float("nan"))
    mia = m.get("mia", {}).get("attack_auroc", float("nan"))
    print(f"{name:<16}{auroc:>11.4f}{dice:>10.4f}{w:>11.4f}{ks:>8.4f}{mia:>9.4f}")

# --- 6. FedEraser post-replay sanity (expected to be far off — confirms non-transfer) ---
try:
    import torch
    from models.ae_2d import AE2D, reconstruction_loss
    from data.datasets import get_client_loader
    from federated.client import set_parameters
    dev = "cuda" if torch.cuda.is_available() else "cpu"; mm = cfg["model"]
    def recon_of(p, cid=1, nb=10):
        ck = torch.load(p, weights_only=False)
        net = AE2D(mm["in_channels"], mm["base_channels"], mm["latent_channels"]).to(dev)
        set_parameters(net, [t.numpy() for t in ck["parameters"]]); net.eval()
        dl = get_client_loader(Path(PROCESSED_LOCAL)/f"client_{cid}", "val", "healthy", 32, False)
        tot=n=0
        with torch.no_grad():
            for i,(x,_) in enumerate(dl):
                if i>=nb: break
                x=x.to(dev); tot+=reconstruction_loss(net(x),x,mm["loss"]).item(); n+=1
        return tot/max(n,1)
    base_ck = f"{base_local}/round_0100/global_model.pt"
    pr = f"{OUT}/unlearn_fed_eraser_forgot_{FORGOTTEN_CLIENT}/post_replay_model.pt"
    print(f"\n[sanity] base(round100) recon = {recon_of(base_ck):.4f}")
    if Path(pr).exists():
        print(f"[sanity] FedEraser post-replay recon = {recon_of(pr):.4f}  "
              f"(far above base = replay doesn't transfer, as expected)")
except Exception as e:
    print("sanity check skipped:", e)

# --- 7. back up THIS seed's outputs to Drive ---
Path(OUTPUTS_DRIVE).mkdir(parents=True, exist_ok=True)
shutil.make_archive(f"/content/outputs_{RUN_TAG}", "zip", OUT)
shutil.copy2(f"/content/outputs_{RUN_TAG}.zip", f"{OUTPUTS_DRIVE}/outputs_{RUN_TAG}.zip")
print(f"\nBacked up to {OUTPUTS_DRIVE}/outputs_{RUN_TAG}.zip")
print(f"Total run time: {(time.time()-t0)/60:.1f} min")

Streaming output truncated to the last 5000 lines.
INFO :      	               (70, 0.7809557028252528),
INFO :      	               (71, 0.7809557028252528),
INFO :      	               (72, 0.780955702825253),
INFO :      	               (73, 0.7809557028252528),
INFO :      	               (74, 0.7809557028252528),
INFO :      	               (75, 0.780955702825253),
INFO :      	               (76, 0.7809557028252528),
INFO :      	               (77, 0.780955702825253),
INFO :      	               (78, 0.7809557028252528),
INFO :      	               (79, 0.7809557028252528),
INFO :      	               (80, 0.7809557028252528),
INFO :      	               (81, 0.780955702825253),
INFO :      	               (82, 0.7809557028252528),
INFO :      	               (83, 0.7809557028252528),
INFO :      	               (84, 0.7809557028252528),
INFO :      	               (85, 0.7809557028252528),
INFO :      	               (86, 0.7809557028252528),
INFO :      	               (87, 0.

In [ ]:
import os
for s in (42, 123, 2024):
    p = f"/content/outputs_seed{s}/base_federation/round_0100/global_model.pt"
    print(f"seed {s} base federation present: {os.path.exists(p)}")

seed 42 base federation present: False
seed 123 base federation present: True
seed 2024 base federation present: True


In [ ]:
# verified backup: zip, copy, then test the copy opens before trusting it
import shutil, zipfile
shutil.make_archive(f"/content/outputs_{RUN_TAG}", "zip", OUT)
dst = f"{OUTPUTS_DRIVE}/outputs_{RUN_TAG}.zip"
shutil.copy2(f"/content/outputs_{RUN_TAG}.zip", dst)
try:
    with zipfile.ZipFile(dst) as z:
        assert z.testzip() is None
    print("backup verified OK:", dst)
except Exception as e:
    print("WARNING: backup did not verify —", e, "\nKeep the local copy:", f"/content/outputs_{RUN_TAG}.zip")

backup verified OK: /content/drive/MyDrive/fed-unlearn-project/outputs_rewrite/outputs_seed2024_forget0.zip


In [ ]:
SEED = 123
FORGOTTEN_CLIENT = 2

import sys, subprocess, json, yaml, time, shutil, zipfile
from pathlib import Path
for n in ["CODE_LOCAL","PROCESSED_LOCAL","BASE_CONFIG","OUTPUTS_DRIVE"]:
    assert n in dir(), f"{n} not defined — run sections 3 and 8 first."
OUT = f"/content/outputs_seed{SEED}"; Path(OUT).mkdir(parents=True, exist_ok=True)
def run_stream(args):
    print("$"," ".join(args)); sys.stdout.flush()
    p=subprocess.Popen(args,cwd=CODE_LOCAL,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in p.stdout: print(line,end=""); sys.stdout.flush()
    p.wait()
    if p.returncode: raise RuntimeError(f"FAILED ({p.returncode}): {' '.join(args)}")
with open(BASE_CONFIG) as f: cfg=yaml.safe_load(f)
cfg["experiment"]["seed"]=SEED; cfg["experiment"]["output_dir"]=OUT
cfg["data"]["processed_dir"]=PROCESSED_LOCAL; cfg["unlearning"]["forgotten_client"]=FORGOTTEN_CLIENT
cfg["unlearning"]["fed_eraser"]["replay_stride"]=1
RUN_CFG=f"/content/config_seed{SEED}_forget{FORGOTTEN_CLIENT}.yaml"
with open(RUN_CFG,"w") as f: yaml.safe_dump(cfg,f,sort_keys=False)
RUN_TAG=f"seed{SEED}_forget{FORGOTTEN_CLIENT}"; t0=time.time()

base_local=f"{OUT}/base_federation"
if Path(f"{base_local}/round_0100/global_model.pt").exists():
    print(">>> base present — skipping (fast run).\n")
else:
    print(">>> base MISSING — retraining (~97 min)...\n")
    run_stream(["python","scripts/run_base_federation.py","--config",RUN_CFG])
assert Path(f"{base_local}/round_0100/global_model.pt").exists(),"base missing"

oracle_local=f"{OUT}/oracle_forgot_{FORGOTTEN_CLIENT}"
if not Path(f"{oracle_local}/round_0100/global_model.pt").exists():
    print(">>> oracle (client 2)...\n")
    run_stream(["python","scripts/run_oracle.py","--config",RUN_CFG,"--forgotten_client",str(FORGOTTEN_CLIENT)])
print(">>> unlearning...\n")
run_stream(["python","scripts/run_unlearning.py","--config",RUN_CFG,"--forgotten_client",str(FORGOTTEN_CLIENT),"--base_run_dir",base_local])
print(">>> evaluation...\n")
manifest=f"{OUT}/manifest_forgot_{FORGOTTEN_CLIENT}.json"
run_stream(["python","scripts/run_evaluation.py","--config",RUN_CFG,"--forgotten_client",str(FORGOTTEN_CLIENT),"--manifest",manifest])

res=json.load(open(f"{OUT}/evaluation_results_forgot_{FORGOTTEN_CLIENT}.json"))
print(f"\n=== seed {SEED}, forgotten client {FORGOTTEN_CLIENT} ===")
print(f"{'method':<16}{'AUROC_rem':>11}{'Dice_rem':>10}{'W_oracle':>11}{'KS':>8}{'MIA':>9}")
for name in ["pre_unlearning","oracle","grad_ascent","kd","fed_eraser"]:
    m=res["metrics"].get(name,{}); ad=m.get("anomaly_detection",{}).get("remaining",{}).get("test",{}); eff=m.get("efficacy",{})
    a=ad.get("auroc",float("nan")); d=ad.get("dice",float("nan"))
    w=0.0 if name=="oracle" else eff.get("wasserstein",float("nan"))
    ks=0.0 if name=="oracle" else eff.get("ks_statistic",float("nan"))
    mia=m.get("mia",{}).get("attack_auroc",float("nan"))
    print(f"{name:<16}{a:>11.4f}{d:>10.4f}{w:>11.4f}{ks:>8.4f}{mia:>9.4f}")

# verified backup
shutil.make_archive(f"/content/outputs_{RUN_TAG}","zip",OUT)
dst=f"{OUTPUTS_DRIVE}/outputs_{RUN_TAG}.zip"; shutil.copy2(f"/content/outputs_{RUN_TAG}.zip",dst)
try:
    with zipfile.ZipFile(dst) as z: assert z.testzip() is None
    print("\nbackup verified OK:",dst)
except Exception as e:
    print("\nWARNING backup unverified:",e,"| local copy:",f"/content/outputs_{RUN_TAG}.zip")
print(f"Total: {(time.time()-t0)/60:.1f} min")

>>> base present — skipping (fast run).

>>> oracle (client 2)...

$ python scripts/run_oracle.py --config /content/config_seed123_forget2.yaml --forgotten_client 2
2026-06-19 15:23:00.937928: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-19 15:23:01.038066: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a new Flower app from a template

		$ flwr run  # Run the Flower app

In [ ]:
SEED = 2024
FORGOTTEN_CLIENT = 2

import sys, subprocess, json, yaml, time, shutil, zipfile
from pathlib import Path
for n in ["CODE_LOCAL","PROCESSED_LOCAL","BASE_CONFIG","OUTPUTS_DRIVE"]:
    assert n in dir(), f"{n} not defined — run sections 3 and 8 first."
OUT = f"/content/outputs_seed{SEED}"; Path(OUT).mkdir(parents=True, exist_ok=True)
def run_stream(args):
    print("$"," ".join(args)); sys.stdout.flush()
    p=subprocess.Popen(args,cwd=CODE_LOCAL,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in p.stdout: print(line,end=""); sys.stdout.flush()
    p.wait()
    if p.returncode: raise RuntimeError(f"FAILED ({p.returncode}): {' '.join(args)}")
with open(BASE_CONFIG) as f: cfg=yaml.safe_load(f)
cfg["experiment"]["seed"]=SEED; cfg["experiment"]["output_dir"]=OUT
cfg["data"]["processed_dir"]=PROCESSED_LOCAL; cfg["unlearning"]["forgotten_client"]=FORGOTTEN_CLIENT
cfg["unlearning"]["fed_eraser"]["replay_stride"]=1
RUN_CFG=f"/content/config_seed{SEED}_forget{FORGOTTEN_CLIENT}.yaml"
with open(RUN_CFG,"w") as f: yaml.safe_dump(cfg,f,sort_keys=False)
RUN_TAG=f"seed{SEED}_forget{FORGOTTEN_CLIENT}"; t0=time.time()

base_local=f"{OUT}/base_federation"
if Path(f"{base_local}/round_0100/global_model.pt").exists():
    print(">>> base present — skipping (fast run).\n")
else:
    print(">>> base MISSING — retraining (~97 min)...\n")
    run_stream(["python","scripts/run_base_federation.py","--config",RUN_CFG])
assert Path(f"{base_local}/round_0100/global_model.pt").exists(),"base missing"

oracle_local=f"{OUT}/oracle_forgot_{FORGOTTEN_CLIENT}"
if not Path(f"{oracle_local}/round_0100/global_model.pt").exists():
    print(">>> oracle (client 2)...\n")
    run_stream(["python","scripts/run_oracle.py","--config",RUN_CFG,"--forgotten_client",str(FORGOTTEN_CLIENT)])
print(">>> unlearning...\n")
run_stream(["python","scripts/run_unlearning.py","--config",RUN_CFG,"--forgotten_client",str(FORGOTTEN_CLIENT),"--base_run_dir",base_local])
print(">>> evaluation...\n")
manifest=f"{OUT}/manifest_forgot_{FORGOTTEN_CLIENT}.json"
run_stream(["python","scripts/run_evaluation.py","--config",RUN_CFG,"--forgotten_client",str(FORGOTTEN_CLIENT),"--manifest",manifest])

res=json.load(open(f"{OUT}/evaluation_results_forgot_{FORGOTTEN_CLIENT}.json"))
print(f"\n=== seed {SEED}, forgotten client {FORGOTTEN_CLIENT} ===")
print(f"{'method':<16}{'AUROC_rem':>11}{'Dice_rem':>10}{'W_oracle':>11}{'KS':>8}{'MIA':>9}")
for name in ["pre_unlearning","oracle","grad_ascent","kd","fed_eraser"]:
    m=res["metrics"].get(name,{}); ad=m.get("anomaly_detection",{}).get("remaining",{}).get("test",{}); eff=m.get("efficacy",{})
    a=ad.get("auroc",float("nan")); d=ad.get("dice",float("nan"))
    w=0.0 if name=="oracle" else eff.get("wasserstein",float("nan"))
    ks=0.0 if name=="oracle" else eff.get("ks_statistic",float("nan"))
    mia=m.get("mia",{}).get("attack_auroc",float("nan"))
    print(f"{name:<16}{a:>11.4f}{d:>10.4f}{w:>11.4f}{ks:>8.4f}{mia:>9.4f}")

# verified backup
shutil.make_archive(f"/content/outputs_{RUN_TAG}","zip",OUT)
dst=f"{OUTPUTS_DRIVE}/outputs_{RUN_TAG}.zip"; shutil.copy2(f"/content/outputs_{RUN_TAG}.zip",dst)
try:
    with zipfile.ZipFile(dst) as z: assert z.testzip() is None
    print("\nbackup verified OK:",dst)
except Exception as e:
    print("\nWARNING backup unverified:",e,"| local copy:",f"/content/outputs_{RUN_TAG}.zip")
print(f"Total: {(time.time()-t0)/60:.1f} min")

>>> base present — skipping (fast run).

>>> oracle (client 2)...

$ python scripts/run_oracle.py --config /content/config_seed2024_forget2.yaml --forgotten_client 2
2026-06-19 16:52:48.753105: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-19 16:52:49.911181: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a new Flower app from a template

		$ flwr run  # Run the Flower ap

In [ ]:
SEED = 42
FORGOTTEN_CLIENT = 2

import sys, subprocess, json, yaml, time, shutil, zipfile
from pathlib import Path
for n in ["CODE_LOCAL","PROCESSED_LOCAL","BASE_CONFIG","OUTPUTS_DRIVE"]:
    assert n in dir(), f"{n} not defined — run sections 3 and 8 first."
OUT = f"/content/outputs_seed{SEED}"; Path(OUT).mkdir(parents=True, exist_ok=True)
def run_stream(args):
    print("$"," ".join(args)); sys.stdout.flush()
    p=subprocess.Popen(args,cwd=CODE_LOCAL,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in p.stdout: print(line,end=""); sys.stdout.flush()
    p.wait()
    if p.returncode: raise RuntimeError(f"FAILED ({p.returncode}): {' '.join(args)}")
with open(BASE_CONFIG) as f: cfg=yaml.safe_load(f)
cfg["experiment"]["seed"]=SEED; cfg["experiment"]["output_dir"]=OUT
cfg["data"]["processed_dir"]=PROCESSED_LOCAL; cfg["unlearning"]["forgotten_client"]=FORGOTTEN_CLIENT
cfg["unlearning"]["fed_eraser"]["replay_stride"]=1
RUN_CFG=f"/content/config_seed{SEED}_forget{FORGOTTEN_CLIENT}.yaml"
with open(RUN_CFG,"w") as f: yaml.safe_dump(cfg,f,sort_keys=False)
RUN_TAG=f"seed{SEED}_forget{FORGOTTEN_CLIENT}"; t0=time.time()

base_local=f"{OUT}/base_federation"
if Path(f"{base_local}/round_0100/global_model.pt").exists():
    print(">>> base present — skipping (fast run).\n")
else:
    print(">>> base MISSING — retraining (~97 min)...\n")
    run_stream(["python","scripts/run_base_federation.py","--config",RUN_CFG])
assert Path(f"{base_local}/round_0100/global_model.pt").exists(),"base missing"

oracle_local=f"{OUT}/oracle_forgot_{FORGOTTEN_CLIENT}"
if not Path(f"{oracle_local}/round_0100/global_model.pt").exists():
    print(">>> oracle (client 2)...\n")
    run_stream(["python","scripts/run_oracle.py","--config",RUN_CFG,"--forgotten_client",str(FORGOTTEN_CLIENT)])
print(">>> unlearning...\n")
run_stream(["python","scripts/run_unlearning.py","--config",RUN_CFG,"--forgotten_client",str(FORGOTTEN_CLIENT),"--base_run_dir",base_local])
print(">>> evaluation...\n")
manifest=f"{OUT}/manifest_forgot_{FORGOTTEN_CLIENT}.json"
run_stream(["python","scripts/run_evaluation.py","--config",RUN_CFG,"--forgotten_client",str(FORGOTTEN_CLIENT),"--manifest",manifest])

res=json.load(open(f"{OUT}/evaluation_results_forgot_{FORGOTTEN_CLIENT}.json"))
print(f"\n=== seed {SEED}, forgotten client {FORGOTTEN_CLIENT} ===")
print(f"{'method':<16}{'AUROC_rem':>11}{'Dice_rem':>10}{'W_oracle':>11}{'KS':>8}{'MIA':>9}")
for name in ["pre_unlearning","oracle","grad_ascent","kd","fed_eraser"]:
    m=res["metrics"].get(name,{}); ad=m.get("anomaly_detection",{}).get("remaining",{}).get("test",{}); eff=m.get("efficacy",{})
    a=ad.get("auroc",float("nan")); d=ad.get("dice",float("nan"))
    w=0.0 if name=="oracle" else eff.get("wasserstein",float("nan"))
    ks=0.0 if name=="oracle" else eff.get("ks_statistic",float("nan"))
    mia=m.get("mia",{}).get("attack_auroc",float("nan"))
    print(f"{name:<16}{a:>11.4f}{d:>10.4f}{w:>11.4f}{ks:>8.4f}{mia:>9.4f}")

# verified backup
shutil.make_archive(f"/content/outputs_{RUN_TAG}","zip",OUT)
dst=f"{OUTPUTS_DRIVE}/outputs_{RUN_TAG}.zip"; shutil.copy2(f"/content/outputs_{RUN_TAG}.zip",dst)
try:
    with zipfile.ZipFile(dst) as z: assert z.testzip() is None
    print("\nbackup verified OK:",dst)
except Exception as e:
    print("\nWARNING backup unverified:",e,"| local copy:",f"/content/outputs_{RUN_TAG}.zip")
print(f"Total: {(time.time()-t0)/60:.1f} min")

Streaming output truncated to the last 5000 lines.
INFO :      	               (79, 0.7809557028252528),
INFO :      	               (80, 0.7809557028252528),
INFO :      	               (81, 0.7809557028252528),
INFO :      	               (82, 0.780955702825253),
INFO :      	               (83, 0.7809557028252528),
INFO :      	               (84, 0.7809557028252528),
INFO :      	               (85, 0.7809557028252528),
INFO :      	               (86, 0.7809557028252528),
INFO :      	               (87, 0.780955702825253),
INFO :      	               (88, 0.780955702825253),
INFO :      	               (89, 0.7809557028252527),
INFO :      	               (90, 0.7809557028252528),
INFO :      	               (91, 0.780955702825253),
INFO :      	               (92, 0.7809557028252527),
INFO :      	               (93, 0.780955702825253),
INFO :      	               (94, 0.780955702825253),
INFO :      	               (95, 0.7809557028252528),
INFO :      	               (96, 0.78

---
## Push-weight ablation

From here on the cells explore scaling the distillation push weight by a
measured memorisation signal. The paper makes no adaptivity claim and reports
only the direct `alpha_p = 0` ablation in Table 2. The `kd_fraction`,
`kd_teacher_signal`, `kd_teacher_gap` and `kd_oracle_divergence` variants below
are exploratory and are not in the paper.

In [ ]:
import yaml, os
SEED, CLIENT = 123, 2
OUT = f"/content/outputs_seed{SEED}"
BASE_CONFIG = f"{CODE_LOCAL}/configs/base.yaml"

cfg = yaml.safe_load(open(BASE_CONFIG))
cfg["experiment"]["seed"] = SEED
cfg["experiment"]["output_dir"] = OUT
cfg["data"]["processed_dir"] = PROCESSED_LOCAL
cfg["unlearning"]["forgotten_client"] = CLIENT
RUN_CFG = f"/content/cfg_optC_s{SEED}_c{CLIENT}.yaml"
yaml.safe_dump(cfg, open(RUN_CFG, "w"), sort_keys=False)

# hard guard: never silently retrain
assert os.path.exists(f"{OUT}/base_federation/round_0100/global_model.pt"), "base missing - STOP"
assert os.path.exists(f"{OUT}/oracle_forgot_{CLIENT}/round_0100/global_model.pt"), "oracle missing - STOP"

run_stream(["python", "scripts/run_unlearning.py",
            "--config", RUN_CFG, "--forgotten_client", str(CLIENT),
            "--base_run_dir", f"{OUT}/base_federation",
            "--methods", "kd", "kd_fraction", "kd_teacher_signal",
                         "kd_teacher_gap", "kd_oracle_divergence"])

$ python scripts/run_unlearning.py --config /content/cfg_optC_s123_c2.yaml --forgotten_client 2 --base_run_dir /content/outputs_seed123/base_federation --methods kd kd_fraction kd_teacher_signal kd_teacher_gap kd_oracle_divergence
2026-08-03 20:17:28.523774: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-03 20:17:28.594476: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a 

In [ ]:
import json, glob
for f in sorted(glob.glob(f"{OUT}/unlearn_kd*_forgot_{CLIENT}/push_scaling.json")):
    d = json.load(open(f))
    extra = ""
    if "memorisation_signal" in d:
        extra = f"  s={d['memorisation_signal']:+.4f}"
        if "e_forgotten" in d:
            extra += f"  (e_forgot={d['e_forgotten']:.5f}, e_remain={d['e_remaining']:.5f})"
        if "e_train" in d:
            extra += f"  (e_train={d['e_train']:.5f}, e_held={d['e_heldout']:.5f})"
    print(f"{d['mode']:<18} scale={d['scale']:.3f}{extra}")

fixed              scale=1.000
fraction           scale=0.053
oracle_divergence  scale=0.084
teacher_gap        scale=0.000  s=-0.1466  (e_train=0.01663, e_held=0.01450)
teacher_signal     scale=0.071  s=+0.0071  (e_forgot=0.01663, e_remain=0.01674)


In [ ]:
run_stream(["python", "scripts/run_evaluation.py", "--config", RUN_CFG,
            "--forgotten_client", str(CLIENT),
            "--manifest", f"{OUT}/manifest_forgot_{CLIENT}.json"])

$ python scripts/run_evaluation.py --config /content/cfg_optC_s123_c2.yaml --forgotten_client 2 --manifest /content/outputs_seed123/manifest_forgot_2.json

EVALUATION 1/3: Anomaly Detection (pixel AUROC, Dice)

  pre_unlearning: /content/outputs_seed123/base_federation/round_0100/global_model.pt
    remaining: AUROC=0.8756, Dice=0.2188
    forgotten: AUROC=0.8609, Dice=0.2113

  oracle: /content/outputs_seed123/oracle_forgot_2/round_0100/global_model.pt
    remaining: AUROC=0.8682, Dice=0.1990
    forgotten: AUROC=0.8552, Dice=0.2003

  grad_ascent: /content/outputs_seed123/unlearn_grad_ascent_forgot_2/final_unlearned_model.pt
    remaining: AUROC=0.8545, Dice=0.1707
    forgotten: AUROC=0.8432, Dice=0.1866

  kd: /content/outputs_seed123/unlearn_kd_forgot_2/final_unlearned_model.pt
    remaining: AUROC=0.9526, Dice=0.4796
    forgotten: AUROC=0.9319, Dice=0.4280

  fed_eraser: /content/outputs_seed123/unlearn_fed_eraser_forgot_2/final_unlearned_model.pt
    remaining: AUROC=0.8553, Di

### Overnight batch

This batch reran the ablation across the remaining cells. For seed 42 client 0
it also retrained the oracle, which had been lost with the corrupt archive. It
did not retrain that cell's base federation and did not run FedEraser for it,
which is why the slim `results_seed42_forget0.zip` archive carries no FedEraser
row.

The job list below holds five of the six cells. Seed 123, client 2 is absent
from it, so no `results_seed123_forget2.zip` was written. That cell's ablation
was run separately and its slim archive was not kept, which is why the
aggregation in the next cell covers five cells and not six.

In [ ]:
# ============================================================================
# Ablation across remaining (seed, client) cells
#

import os, sys, json, time, shutil, zipfile, subprocess, traceback
from pathlib import Path

# ---------------------------------------------------------------- config ---
PROGRESS   = f"{OUTPUTS_DRIVE}/BATCH_PROGRESS.json"
LOGDIR_DRV = f"{OUTPUTS_DRIVE}/logs"
Path(LOGDIR_DRV).mkdir(parents=True, exist_ok=True)

KD_VARIANTS = ["kd", "kd_fraction", "kd_teacher_signal",
               "kd_teacher_gap", "kd_oracle_divergence"]

# (seed, client, source_zip, needs_oracle_training, methods)
JOBS = [
    (123,  0, "outputs_seed123_forget2.zip",  False, KD_VARIANTS),
    (2024, 0, "outputs_seed2024_forget2.zip", False, KD_VARIANTS),
    (2024, 2, "outputs_seed2024_forget2.zip", False, KD_VARIANTS),
    (42,   2, "outputs_seed42_forget2.zip",   False, KD_VARIANTS),
    # seed 42 client 0: oracle was lost with the corrupt backup -> train it.
    # grad_ascent added because this cell has no prior baselines at all.
    (42,   0, "outputs_seed42_forget2.zip",   True,  ["grad_ascent"] + KD_VARIANTS),
]

# ---------------------------------------------------------------- helpers --
def load_progress():
    if os.path.exists(PROGRESS):
        try:
            return json.load(open(PROGRESS))
        except Exception:
            pass
    return {}

def save_progress(p):
    tmp = "/content/_progress.json"
    json.dump(p, open(tmp, "w"), indent=2)
    shutil.copy2(tmp, PROGRESS)

def log(msg, logfile=None):
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(line, flush=True)
    if logfile:
        with open(logfile, "a") as f:
            f.write(line + "\n")

def run_logged(args, logfile, cwd):
    """Run a command, streaming into a log file. Returns True on success."""
    with open(logfile, "a") as f:
        f.write(f"\n$ {' '.join(args)}\n")
        f.flush()
        p = subprocess.Popen(args, cwd=cwd, stdout=f, stderr=subprocess.STDOUT, text=True)
        p.wait()
    return p.returncode == 0

def free_gb():
    return shutil.disk_usage("/content")[2] / 1e9

def extract_seed(seed, zipname, logfile):
    """Ensure /content/outputs_seed{seed} holds a complete base federation."""
    out = f"/content/outputs_seed{seed}"
    ok = os.path.exists(f"{out}/base_federation/round_0100/global_model.pt")
    if ok:
        log(f"  seed {seed} already extracted", logfile)
        return out
    # clear other seeds to free disk
    for d in Path("/content").glob("outputs_seed*"):
        if d.name != f"outputs_seed{seed}":
            shutil.rmtree(d, ignore_errors=True)
            log(f"  cleared {d.name} to free disk", logfile)
    shutil.rmtree(out, ignore_errors=True)
    log(f"  extracting {zipname} (free: {free_gb():.1f} GB)...", logfile)
    t0 = time.time()
    with zipfile.ZipFile(f"{OUTPUTS_DRIVE}/{zipname}") as z:
        z.extractall(out)
    log(f"  extracted in {(time.time()-t0)/60:.1f} min (free now: {free_gb():.1f} GB)", logfile)
    return out

def slim_backup(out, seed, client, tag, logfile):
    """Archive only the small artefacts needed for analysis, and verify it."""
    stage = f"/content/_slim_{tag}"
    shutil.rmtree(stage, ignore_errors=True)
    Path(stage).mkdir(parents=True, exist_ok=True)
    picked = 0
    for name in [f"evaluation_results_forgot_{client}.json",
                 f"manifest_forgot_{client}.json"]:
        src = Path(out) / name
        if src.exists():
            shutil.copy2(src, Path(stage) / name); picked += 1
    for d in Path(out).glob(f"unlearn_*_forgot_{client}"):
        dst = Path(stage) / d.name
        dst.mkdir(parents=True, exist_ok=True)
        for fn in ["push_scaling.json", "distillation_log.json",
                   "final_unlearned_model.pt"]:
            if (d / fn).exists():
                shutil.copy2(d / fn, dst / fn); picked += 1
    arch = f"/content/results_{tag}"
    shutil.make_archive(arch, "zip", stage)
    dst = f"{OUTPUTS_DRIVE}/results_{tag}.zip"
    shutil.copy2(arch + ".zip", dst)
    try:                                   # verify (we had a corrupt zip before)
        with zipfile.ZipFile(dst) as z:
            assert z.testzip() is None
        log(f"  backup verified: results_{tag}.zip ({picked} files, "
            f"{os.path.getsize(dst)/1e6:.1f} MB)", logfile)
        return True
    except Exception as e:
        log(f"  !! BACKUP FAILED VERIFICATION: {e}", logfile)
        return False

def summarise(out, client, logfile):
    """Print the scales and the results table into the log and stdout."""
    lines = []
    for f in sorted(Path(out).glob(f"unlearn_kd*_forgot_{client}/push_scaling.json")):
        d = json.load(open(f))
        extra = ""
        if "memorisation_signal" in d:
            extra = f"  s={d['memorisation_signal']:+.4f}"
        lines.append(f"    {d['mode']:<20} scale={d['scale']:.3f}{extra}")
    rp = Path(out) / f"evaluation_results_forgot_{client}.json"
    if rp.exists():
        res = json.load(open(rp))["metrics"]
        lines.append(f"    {'method':<22}{'AUROC':>9}{'W1':>10}{'KS':>9}{'MIA':>9}")
        for name in ["pre_unlearning", "oracle", "grad_ascent", "kd",
                     "kd_fraction", "kd_teacher_signal", "kd_teacher_gap",
                     "kd_oracle_divergence", "fed_eraser"]:
            m = res.get(name)
            if not m:
                continue
            ad = m.get("anomaly_detection", {}).get("remaining", {}).get("test", {})
            ef = m.get("efficacy", {})
            a = ad.get("auroc", float("nan"))
            w = 0.0 if name == "oracle" else ef.get("wasserstein", float("nan"))
            k = 0.0 if name == "oracle" else ef.get("ks_statistic", float("nan"))
            mi = m.get("mia", {}).get("attack_auroc", float("nan"))
            lines.append(f"    {name:<22}{a:>9.4f}{w:>10.4f}{k:>9.4f}{mi:>9.4f}")
    for L in lines:
        log(L, logfile)

# ---------------------------------------------------------------- batch ----
prog = load_progress()
batch_t0 = time.time()
master_log = f"/content/overnight_{time.strftime('%Y%m%d_%H%M')}.log"

log("=" * 68, master_log)
log(f"OVERNIGHT BATCH START — {len(JOBS)} jobs, {free_gb():.1f} GB free", master_log)
log("=" * 68, master_log)

for jn, (seed, client, zipname, need_oracle, methods) in enumerate(JOBS, 1):
    tag = f"seed{seed}_forget{client}"
    head = f"JOB {jn}/{len(JOBS)}: seed {seed}, client {client}"

    if prog.get(tag, {}).get("status") == "done":
        log(f"\n{head} — already done, skipping", master_log)
        continue

    jlog = f"/content/log_{tag}.log"
    log("\n" + "-" * 68, master_log)
    log(head + (" [oracle training required]" if need_oracle else ""), master_log)
    log("-" * 68, master_log)
    t0 = time.time()

    try:
        OUT = extract_seed(seed, zipname, master_log)

        # ---- per-run config ----
        import yaml
        cfg = yaml.safe_load(open(f"{CODE_LOCAL}/configs/base.yaml"))
        cfg["experiment"]["seed"] = seed
        cfg["experiment"]["output_dir"] = OUT
        cfg["data"]["processed_dir"] = PROCESSED_LOCAL
        cfg["unlearning"]["forgotten_client"] = client
        RUN_CFG = f"/content/cfg_{tag}.yaml"
        yaml.safe_dump(cfg, open(RUN_CFG, "w"), sort_keys=False)

        # ---- guard: base must exist, never silently retrain it ----
        base_ck = f"{OUT}/base_federation/round_0100/global_model.pt"
        if not os.path.exists(base_ck):
            raise FileNotFoundError(f"base federation missing at {base_ck}")

        # ---- oracle ----
        oracle_ck = f"{OUT}/oracle_forgot_{client}/round_0100/global_model.pt"
        if not os.path.exists(oracle_ck):
            if not need_oracle:
                raise FileNotFoundError(
                    f"oracle missing at {oracle_ck} and this job was not "
                    f"flagged for oracle training")
            log("  training oracle (~90 min)...", master_log)
            if not run_logged(["python", "scripts/run_oracle.py",
                               "--config", RUN_CFG,
                               "--forgotten_client", str(client)],
                              jlog, CODE_LOCAL):
                raise RuntimeError("oracle training failed")
            if not os.path.exists(oracle_ck):
                raise RuntimeError("oracle finished but checkpoint absent")
            log("  oracle done", master_log)

        # ---- unlearning ablation ----
        log(f"  running {len(methods)} methods...", master_log)
        if not run_logged(["python", "scripts/run_unlearning.py",
                           "--config", RUN_CFG,
                           "--forgotten_client", str(client),
                           "--base_run_dir", f"{OUT}/base_federation",
                           "--methods"] + methods, jlog, CODE_LOCAL):
            raise RuntimeError("unlearning failed")

        # ---- evaluation ----
        log("  evaluating...", master_log)
        if not run_logged(["python", "scripts/run_evaluation.py",
                           "--config", RUN_CFG,
                           "--forgotten_client", str(client),
                           "--manifest", f"{OUT}/manifest_forgot_{client}.json"],
                          jlog, CODE_LOCAL):
            raise RuntimeError("evaluation failed")

        summarise(OUT, client, master_log)
        backed = slim_backup(OUT, seed, client, tag, master_log)

        prog[tag] = {"status": "done", "minutes": round((time.time() - t0) / 60, 1),
                     "backup_verified": backed,
                     "finished": time.strftime("%Y-%m-%d %H:%M")}
        log(f"  JOB COMPLETE in {(time.time()-t0)/60:.1f} min", master_log)

    except Exception as e:
        prog[tag] = {"status": "failed", "error": str(e),
                     "minutes": round((time.time() - t0) / 60, 1),
                     "finished": time.strftime("%Y-%m-%d %H:%M")}
        log(f"  JOB FAILED: {e}", master_log)
        with open(master_log, "a") as f:
            f.write(traceback.format_exc())

    save_progress(prog)
    for lf in [master_log, jlog]:
        if os.path.exists(lf):
            shutil.copy2(lf, f"{LOGDIR_DRV}/{os.path.basename(lf)}")

# ---------------------------------------------------------------- summary --
log("\n" + "=" * 68, master_log)
log(f"BATCH FINISHED in {(time.time()-batch_t0)/60:.0f} min", master_log)
for jn, (seed, client, *_rest) in enumerate(JOBS, 1):
    tag = f"seed{seed}_forget{client}"
    st = prog.get(tag, {})
    log(f"  seed {seed} / client {client}: {st.get('status','not run'):<8} "
        f"{st.get('minutes','-')} min  {st.get('error','')}", master_log)
log("=" * 68, master_log)
shutil.copy2(master_log, f"{LOGDIR_DRV}/{os.path.basename(master_log)}")
print(f"\nProgress file: {PROGRESS}")
print(f"Logs: {LOGDIR_DRV}")
print(f"Slim results: {OUTPUTS_DRIVE}/results_seed*_forget*.zip")

[04:22:55] ====================================================================
[04:22:55] OVERNIGHT BATCH START — 5 jobs, 67.1 GB free
[04:22:55] ====================================================================
[04:22:55] 
JOB 1/5: seed 123, client 0 — already done, skipping
[04:22:55] 
JOB 2/5: seed 2024, client 0 — already done, skipping
[04:22:55] 
JOB 3/5: seed 2024, client 2 — already done, skipping
[04:22:55] 
JOB 4/5: seed 42, client 2 — already done, skipping
[04:22:55] 
--------------------------------------------------------------------
[04:22:55] JOB 5/5: seed 42, client 0 [oracle training required]
[04:22:55] --------------------------------------------------------------------
[04:22:55]   extracting outputs_seed42_forget2.zip (free: 67.1 GB)...
[04:24:33]   extracted in 1.6 min (free now: 54.9 GB)
[04:24:33]   training oracle (~90 min)...
[04:52:29]   oracle done
[04:52:29]   running 6 methods...
[05:30:35]   evaluating...
[05:44:10]     fixed                scale=1.0

In [ ]:
import json, zipfile, shutil, glob, os, statistics as st
from pathlib import Path

STAGE = "/content/_allresults"
shutil.rmtree(STAGE, ignore_errors=True); Path(STAGE).mkdir(parents=True)

# unpack every slim results archive from Drive
for z in sorted(glob.glob(f"{OUTPUTS_DRIVE}/results_seed*_forget*.zip")):
    tag = Path(z).stem.replace("results_", "")
    d = Path(STAGE) / tag; d.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(z) as zf:
        zf.extractall(d)
    print("unpacked", tag)

METHODS = ["pre_unlearning","oracle","grad_ascent","fed_eraser","kd",
           "kd_fraction","kd_teacher_signal","kd_teacher_gap","kd_oracle_divergence"]
DATA = {}   # (seed, client) -> {method: (auroc, w1, ks, mia)}
SCALE = {}  # (seed, client) -> {mode: (scale, signal)}

for d in sorted(Path(STAGE).iterdir()):
    seed = int(d.name.split("seed")[1].split("_")[0])
    client = int(d.name.split("forget")[1])
    rp = next(d.glob("evaluation_results_forgot_*.json"), None)
    if rp:
        res = json.load(open(rp))["metrics"]
        row = {}
        for m in METHODS:
            if m not in res: continue
            ad = res[m].get("anomaly_detection",{}).get("remaining",{}).get("test",{})
            ef = res[m].get("efficacy",{})
            row[m] = (ad.get("auroc",float("nan")),
                      0.0 if m=="oracle" else ef.get("wasserstein",float("nan")),
                      0.0 if m=="oracle" else ef.get("ks_statistic",float("nan")),
                      res[m].get("mia",{}).get("attack_auroc",float("nan")))
        DATA[(seed,client)] = row
    sc = {}
    for pf in d.glob("unlearn_kd*/push_scaling.json"):
        j = json.load(open(pf))
        sc[j["mode"]] = (j["scale"], j.get("memorisation_signal"))
    SCALE[(seed,client)] = sc

# ---- per-cell tables ----
for key in sorted(DATA, key=lambda k:(k[1],k[0])):
    seed, client = key
    print(f"\n{'='*74}\nseed {seed}, client {client}")
    s = SCALE.get(key,{})
    for mode in ["fixed","fraction","teacher_signal","teacher_gap","oracle_divergence"]:
        if mode in s:
            v,sig = s[mode]
            print(f"   scale[{mode:<18}] = {v:.3f}" + (f"   s={sig:+.4f}" if sig is not None else ""))
    print(f"   {'method':<22}{'AUROC':>9}{'W1':>10}{'KS':>9}{'MIA':>9}")
    for m in METHODS:
        if m in DATA[key]:
            a,w,k,mi = DATA[key][m]
            print(f"   {m:<22}{a:>9.4f}{w:>10.4f}{k:>9.4f}{mi:>9.4f}")

# ---- aggregate across seeds, per client ----
def ms(v): return (st.mean(v), st.stdev(v) if len(v)>1 else 0.0)
for client in (0,2):
    keys = [k for k in DATA if k[1]==client]
    if not keys: continue
    print(f"\n{'='*74}\nCLIENT {client} — mean +/- std over {len(keys)} seeds {sorted(k[0] for k in keys)}")
    print(f"   {'method':<22}{'AUROC':>17}{'W1':>19}{'KS':>17}")
    for m in METHODS:
        vals = [DATA[k][m] for k in keys if m in DATA[k]]
        if len(vals) < 2: continue
        a = ms([v[0] for v in vals]); w = ms([v[1] for v in vals]); ks = ms([v[2] for v in vals])
        print(f"   {m:<22}{a[0]:>9.4f}+/-{a[1]:.4f}{w[0]:>11.4f}+/-{w[1]:.4f}{ks[0]:>9.4f}+/-{ks[1]:.4f}")

unpacked seed123_forget0
unpacked seed2024_forget0
unpacked seed2024_forget2
unpacked seed42_forget0
unpacked seed42_forget2

seed 42, client 0
   scale[fixed             ] = 1.000
   scale[fraction          ] = 1.000
   scale[teacher_signal    ] = 0.000   s=-0.0683
   scale[teacher_gap       ] = 0.000   s=-0.0068
   scale[oracle_divergence ] = 1.000
   method                    AUROC        W1       KS      MIA
   pre_unlearning           0.8583    0.0042   0.4073   0.5010
   oracle                   0.8522    0.0000   0.0000   0.4801
   grad_ascent              0.8622    0.0029   0.2992   0.4986
   kd                       0.9356    0.0060   0.4477   0.4929
   kd_fraction              0.9571    0.0057   0.4268   0.4917
   kd_teacher_signal        0.8603    0.0030   0.2992   0.4945
   kd_teacher_gap           0.8787    0.0021   0.2029   0.4959
   kd_oracle_divergence     0.9550    0.0070   0.4874   0.4901

seed 123, client 0
   scale[fixed             ] = 1.000
   scale[fraction      

### Noise floor

Two configurations that resolve to the same mathematics, both ending at
`alpha_p = 0`, are compared here. They differ by 0.0007 in Wasserstein
distance. That value is the run-to-run noise floor quoted in the paper, and no
claim is made at or below it.

In [ ]:
# 1. Per-seed paired check + noise floor, from the archives you already have
import json, glob
from pathlib import Path
for d in sorted(Path("/content/_allresults").iterdir()):
    seed = int(d.name.split("seed")[1].split("_")[0]); client = int(d.name.split("forget")[1])
    res = json.load(open(next(d.glob("evaluation_results_forgot_*.json"))))["metrics"]
    g = lambda m: res[m]["efficacy"]["wasserstein"] if m in res else float("nan")
    sc = {json.load(open(p))["mode"]: json.load(open(p))["scale"]
          for p in d.glob("unlearn_kd*/push_scaling.json")}
    print(f"seed {seed} client {client}:  gap={g('kd_teacher_gap'):.4f} "
          f"ga={g('grad_ascent'):.4f}  gap_wins={g('kd_teacher_gap')<g('grad_ascent')}"
          f"   [scales: sig={sc.get('teacher_signal')}, gap={sc.get('teacher_gap')}]")

seed 123 client 0:  gap=0.0025 ga=0.0036  gap_wins=True   [scales: sig=0.0, gap=0.0]
seed 2024 client 0:  gap=0.0015 ga=0.0028  gap_wins=True   [scales: sig=0.0, gap=0.0]
seed 2024 client 2:  gap=0.0018 ga=0.0003  gap_wins=False   [scales: sig=0.07328596201951604, gap=0.0]
seed 42 client 0:  gap=0.0021 ga=0.0029  gap_wins=True   [scales: sig=0.0, gap=0.0]
seed 42 client 2:  gap=0.0005 ga=0.0007  gap_wins=True   [scales: sig=0.0, gap=0.0]


In [ ]:
import os, shutil, zipfile
src = "/content/outputs_seed42/oracle_forgot_0"
print("seed42 client-0 oracle on disk:", os.path.exists(f"{src}/round_0100/global_model.pt"))
if os.path.exists(f"{src}/round_0100/global_model.pt"):
    shutil.make_archive("/content/oracle_seed42_forgot0", "zip", src)
    dst = f"{OUTPUTS_DRIVE}/oracle_seed42_forgot0.zip"
    shutil.copy2("/content/oracle_seed42_forgot0.zip", dst)
    with zipfile.ZipFile(dst) as z: assert z.testzip() is None
    print("SAVED + verified:", dst, f"{os.path.getsize(dst)/1e9:.2f} GB")
else:
    print("gone — job 5 will retrain it (~90 min)")

seed42 client-0 oracle on disk: True
SAVED + verified: /content/drive/MyDrive/fed-unlearn-project/outputs_rewrite/oracle_seed42_forgot0.zip 2.40 GB


---
## Determinism investigation

These cells attempt bitwise reproducibility. They did not achieve it. The test count differs from the released suite because the tests
were extended afterwards.

In [ ]:
import shutil, zipfile, subprocess, sys
from pathlib import Path

shutil.rmtree(CODE_LOCAL, ignore_errors=True)
shutil.rmtree("/content/_code_tmp", ignore_errors=True)
with zipfile.ZipFile(CODE_ZIP) as z:
    z.extractall("/content/_code_tmp")
marker = next(Path("/content/_code_tmp").rglob("configs/base.yaml"))
shutil.move(str(marker.parent.parent), CODE_LOCAL)
shutil.rmtree("/content/_code_tmp", ignore_errors=True)
sys.path.insert(0, CODE_LOCAL)

src = open(f"{CODE_LOCAL}/federated/seeding.py").read()
print("has derive_seed:", "def derive_seed" in src)
print("has enable_determinism:", "def enable_determinism" in src)

r = subprocess.run(["python","-m","tests.test_methods"], cwd=CODE_LOCAL,
                   capture_output=True, text=True)
print(r.stdout.strip()[-300:])

has derive_seed: True
has enable_determinism: True
n_clips_scale
PASS test_calibration_zeroes_buffers
PASS test_calibration_excludes_bn_stats
PASS test_kd_clamp_binds
PASS test_seed_determinism
PASS test_memorisation_signal
PASS test_signal_to_scale
PASS test_derive_seed_is_process_stable
PASS test_seeded_shuffle_is_reproducible

12/12 tests passed.


### Determinism check, negative result

This reports a failure. It is kept because the paper's limitations section
rests on it: after sorting the aggregation by client id, replacing a
hash-derived seed with crc32, and enabling deterministic algorithms inside the
Ray workers, a difference remains at round one. That places the source inside a
single client's local training rather than in aggregation.

In [ ]:
import yaml, torch, shutil, os, subprocess, sys

SEED, CLIENT = 123, 2
OUT = f"/content/outputs_seed{SEED}"
cfg = yaml.safe_load(open(f"{CODE_LOCAL}/configs/base.yaml"))
cfg["experiment"].update({"seed": SEED, "output_dir": OUT, "deterministic": True})
cfg["data"]["processed_dir"] = PROCESSED_LOCAL
cfg["unlearning"]["forgotten_client"] = CLIENT
RUN_CFG = "/content/cfg_determinism_check.yaml"
yaml.safe_dump(cfg, open(RUN_CFG,"w"), sort_keys=False)

def run_kd():
    p = subprocess.Popen(["python","scripts/run_unlearning.py","--config",RUN_CFG,
                          "--forgotten_client",str(CLIENT),
                          "--base_run_dir",f"{OUT}/base_federation",
                          "--methods","kd"],
                         cwd=CODE_LOCAL, stdout=subprocess.DEVNULL,
                         stderr=subprocess.STDOUT)
    p.wait(); assert p.returncode == 0, "run failed"

ckpt = f"{OUT}/unlearn_kd_forgot_{CLIENT}/final_unlearned_model.pt"

print("RUN A..."); run_kd(); shutil.copy2(ckpt, "/content/_runA.pt")
print("RUN B..."); run_kd(); shutil.copy2(ckpt, "/content/_runB.pt")

A = torch.load("/content/_runA.pt", weights_only=False)["parameters"]
B = torch.load("/content/_runB.pt", weights_only=False)["parameters"]
identical = all(torch.equal(a,b) for a,b in zip(A,B))
maxdiff = max((a.float()-b.float()).abs().max().item() for a,b in zip(A,B))
print(f"\nBIT-IDENTICAL: {identical}   max weight diff: {maxdiff:.3e}")
print("PASS - noise floor eliminated, proceed to Step 4" if identical
      else "FAIL - a 4th nondeterminism source remains; stop and report this")

RUN A...
RUN B...

BIT-IDENTICAL: False   max weight diff: 2.885e-02
FAIL - a 4th nondeterminism source remains; stop and report this


In [ ]:
srv = open(f"{CODE_LOCAL}/federated/server.py").read()
cli = open(f"{CODE_LOCAL}/federated/client.py").read()
seed = open(f"{CODE_LOCAL}/federated/seeding.py").read()

print("aggregation sorted        :", "results = sorted(" in srv)
print("determinism in AEClient   :", "enable_determinism(True)" in cli)
print("CUBLAS var at import      :", "CUBLAS_WORKSPACE_CONFIG" in seed)
print("use_deterministic_algos   :", "use_deterministic_algorithms" in seed)

aggregation sorted        : False
determinism in AEClient   : False
CUBLAS var at import      : False
use_deterministic_algos   : False


In [ ]:
import shutil, zipfile, sys
from pathlib import Path
shutil.rmtree(CODE_LOCAL, ignore_errors=True)
shutil.rmtree("/content/_code_tmp", ignore_errors=True)
with zipfile.ZipFile(CODE_ZIP) as z: z.extractall("/content/_code_tmp")
m = next(Path("/content/_code_tmp").rglob("configs/base.yaml"))
shutil.move(str(m.parent.parent), CODE_LOCAL)
shutil.rmtree("/content/_code_tmp", ignore_errors=True)
sys.path.insert(0, CODE_LOCAL)
print("restaged. aggregation sorted:",
      "results = sorted(" in open(f"{CODE_LOCAL}/federated/server.py").read())

restaged. aggregation sorted: False


### Where the residual difference sits

The two numbers below, 1.5e-2 at one round and 9.1e-3 at two, are the figures
quoted in the limitations section. The difference does not grow with rounds, so
it is not compounding drift.

In [ ]:
import yaml, torch, shutil, subprocess

def kd_check(rounds):
    cfg = yaml.safe_load(open(f"{CODE_LOCAL}/configs/base.yaml"))
    cfg["experiment"].update({"seed":123, "output_dir":OUT, "deterministic":True})
    cfg["data"]["processed_dir"] = PROCESSED_LOCAL
    cfg["unlearning"]["forgotten_client"] = 2
    cfg["unlearning"]["kd_unlearn"]["distill_rounds"] = rounds
    RC = f"/content/cfg_det_{rounds}.yaml"
    yaml.safe_dump(cfg, open(RC,"w"), sort_keys=False)
    ck = f"{OUT}/unlearn_kd_forgot_2/final_unlearned_model.pt"
    out = []
    for tag in ("A","B"):
        p = subprocess.run(["python","scripts/run_unlearning.py","--config",RC,
                            "--forgotten_client","2","--base_run_dir",
                            f"{OUT}/base_federation","--methods","kd"],
                           cwd=CODE_LOCAL, capture_output=True, text=True)
        assert p.returncode==0, p.stdout[-1500:]
        out.append([t.clone() for t in torch.load(ck, weights_only=False)["parameters"]])
        # surface any nondeterminism warnings PyTorch raised
        for line in p.stdout.split("\n"):
            if "determinis" in line.lower() or "nondeterminis" in line.lower():
                print("   WARN:", line.strip()[:160])
    md = max((a.float()-b.float()).abs().max().item() for a,b in zip(*out))
    print(f"rounds={rounds}: identical={md==0.0}  max_diff={md:.3e}")
    return md

kd_check(1)   # one round: isolates local training
kd_check(2)   # two rounds: isolates aggregation compounding

rounds=1: identical=False  max_diff=1.501e-02
rounds=2: identical=False  max_diff=9.145e-03


0.009145259857177734

In [ ]:
# SAVE COLAB SESSION -> Drive
# Saves everything in /content that is NOT already backed up
import os, json, shutil, zipfile, time
from pathlib import Path

SAVE_DIR = f"{OUTPUTS_DRIVE}/session_save"
Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
manifest = {"saved_at": time.strftime("%Y-%m-%d %H:%M"), "items": []}

def verified_zip(src_dir, name, filt=None):
    """Zip a directory (optionally filtered) to Drive and verify it opens."""
    stage = f"/content/_stage_{name}"
    shutil.rmtree(stage, ignore_errors=True)
    n = 0
    for f in Path(src_dir).rglob("*"):
        if not f.is_file():
            continue
        if filt and not filt(f):
            continue
        rel = f.relative_to(src_dir)
        dst = Path(stage) / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, dst)
        n += 1
    if n == 0:
        print(f"  [skip] {name}: nothing to save")
        shutil.rmtree(stage, ignore_errors=True)
        return
    arch = f"/content/_arch_{name}"
    shutil.make_archive(arch, "zip", stage)
    dst = f"{SAVE_DIR}/{name}.zip"
    shutil.copy2(arch + ".zip", dst)
    try:
        with zipfile.ZipFile(dst) as z:
            assert z.testzip() is None
        mb = os.path.getsize(dst) / 1e6
        print(f"  [ok]   {name}.zip  ({n} files, {mb:.1f} MB) verified")
        manifest["items"].append({"name": name, "files": n, "mb": round(mb, 1)})
    except Exception as e:
        print(f"  [FAIL] {name}: {e}")
    shutil.rmtree(stage, ignore_errors=True)
    os.remove(arch + ".zip")

print("=" * 66)
print("SAVING COLAB SESSION")
print("=" * 66)

# --- 1. Trained ORACLES and BASE final checkpoints ---
# Only the final round is needed to reuse an oracle
print("\n[1] Model checkpoints (final rounds only - small but expensive to retrain)")
for seed_dir in sorted(Path("/content").glob("outputs_seed*")):
    seed = seed_dir.name.replace("outputs_", "")
    def keep_final(f):
        p = str(f)
        return ("round_0100/global_model.pt" in p or
                "round_0001/global_model.pt" in p)
    verified_zip(seed_dir, f"checkpoints_{seed}", keep_final)

# --- 2. ALL unlearning results (models are small; these are today's work) ---
print("\n[2] Unlearning outputs (models + logs + push_scaling)")
for seed_dir in sorted(Path("/content").glob("outputs_seed*")):
    seed = seed_dir.name.replace("outputs_", "")
    def keep_unlearn(f):
        p = str(f)
        return ("unlearn_" in p and f.name in (
                    "final_unlearned_model.pt", "push_scaling.json",
                    "distillation_log.json", "post_replay_model.pt")) \
            or f.name.startswith("evaluation_results_") \
            or f.name.startswith("manifest_")
    verified_zip(seed_dir, f"unlearn_{seed}", keep_unlearn)

# --- 3. Determinism-test artefacts + configs + logs ---
print("\n[3] Diagnostics, configs, logs")
misc = "/content/_misc"
shutil.rmtree(misc, ignore_errors=True); Path(misc).mkdir(parents=True)
for pat in ["/content/_run*.pt", "/content/cfg_*.yaml", "/content/config_*.yaml",
            "/content/*.log", "/content/log_*.log"]:
    import glob
    for f in glob.glob(pat):
        try:
            shutil.copy2(f, Path(misc) / os.path.basename(f))
        except Exception:
            pass
verified_zip(misc, "diagnostics_configs_logs")
shutil.rmtree(misc, ignore_errors=True)

# --- 4. The staged code, exactly as it ran (so results are traceable to it) ---
print("\n[4] Code as it currently stands in the session")
if os.path.exists(CODE_LOCAL):
    verified_zip(CODE_LOCAL, "code_as_run",
                 lambda f: "__pycache__" not in str(f) and f.suffix != ".pyc")

# --- 5. Inventory ---
json.dump(manifest, open("/content/_save_manifest.json", "w"), indent=2)
shutil.copy2("/content/_save_manifest.json", f"{SAVE_DIR}/SAVE_MANIFEST.json")

print("\n" + "=" * 66)
print(f"SAVED TO: {SAVE_DIR}")
for it in manifest["items"]:
    print(f"   {it['name']:<32} {it['files']:>5} files  {it['mb']:>8.1f} MB")
total = sum(i["mb"] for i in manifest["items"])
print(f"   {'TOTAL':<32} {'':>5}        {total:>8.1f} MB")
print("=" * 66)

# --- 6. What is already safe elsewhere (no action needed) ---
print("\nAlready backed up on Drive (not re-saved):")
for f, what in [("processed_p1.zip", "preprocessed slices"),
                ("outputs_rewrite/outputs_seed123_forget2.zip", "seed123 full outputs"),
                ("outputs_rewrite/outputs_seed2024_forget2.zip", "seed2024 full outputs"),
                ("outputs_rewrite/outputs_seed42_forget2.zip", "seed42 full outputs")]:
    p = f"{PROJECT_DIR}/{f}"
    print(f"   {'OK ' if os.path.exists(p) else 'MISSING'} {what}")

SAVING COLAB SESSION

[1] Model checkpoints (final rounds only - small but expensive to retrain)
  [ok]   checkpoints_seed123.zip  (12 files, 48.1 MB) verified
  [ok]   checkpoints_seed42.zip  (15 files, 60.2 MB) verified

[2] Unlearning outputs (models + logs + push_scaling)
  [ok]   unlearn_seed123.zip  (15 files, 32.2 MB) verified
  [ok]   unlearn_seed42.zip  (25 files, 40.4 MB) verified

[3] Diagnostics, configs, logs
  [ok]   diagnostics_configs_logs.zip  (8 files, 8.1 MB) verified

[4] Code as it currently stands in the session
  [ok]   code_as_run.zip  (36 files, 0.1 MB) verified

SAVED TO: /content/drive/MyDrive/fed-unlearn-project/outputs_rewrite/session_save
   checkpoints_seed123                 12 files      48.1 MB
   checkpoints_seed42                  15 files      60.2 MB
   unlearn_seed123                     15 files      32.2 MB
   unlearn_seed42                      25 files      40.4 MB
   diagnostics_configs_logs             8 files       8.1 MB
   code_as_run    

In [ ]:
import zipfile, os
from pathlib import Path

TARGET = "base_federation/round_0100/global_model.pt"
srcs = {123: "outputs_seed123_forget2.zip",
        2024: "outputs_seed2024_forget2.zip",
        42:   "outputs_seed42_forget2.zip"}

for seed, zname in srcs.items():
    out = f"/content/outputs_seed{seed}"
    dst = f"{out}/{TARGET}"
    if os.path.exists(dst):
        print(f"seed {seed}: already present"); continue
    zp = f"{OUTPUTS_DRIVE}/{zname}"
    if not os.path.exists(zp):
        print(f"seed {seed}: ZIP MISSING {zname}"); continue
    with zipfile.ZipFile(zp) as z:
        member = next((m for m in z.namelist() if m.endswith(TARGET)), None)
        if member is None:
            print(f"seed {seed}: target not inside zip"); continue
        Path(dst).parent.mkdir(parents=True, exist_ok=True)
        with z.open(member) as fsrc, open(dst, "wb") as fdst:
            fdst.write(fsrc.read())
    print(f"seed {seed}: extracted ({os.path.getsize(dst)/1e6:.1f} MB)")

print("\nverify:")
for seed in srcs:
    print(f"  seed {seed}:", os.path.exists(f"/content/outputs_seed{seed}/{TARGET}"))

seed 123: extracted (4.3 MB)
seed 2024: extracted (4.3 MB)
seed 42: extracted (4.3 MB)

verify:
  seed 123: True
  seed 2024: True
  seed 42: True


---
## Memorisation signal across clients

This measures whether the memorisation signal varies with client size. It does
not: the estimate is unusable for clients with few held-out cases, and
switching the held-out split reverses its sign for four of six clients. On that
basis the size-aware framing was dropped and the paper makes no adaptivity
claim. `unlearning/push_scaling.py` is retained in the repository for
completeness only.

In [ ]:
# ============================================================================
# DECISIVE EXPERIMENT: does the memorisation signal VARY across clients?
#
# If s is ~0 for all six clients  -> the signal is constant, "size-aware"
#                                    framing is dead, write the validation study.
# If s varies and goes positive   -> the signal is informative, adaptivity is
#     for some clients               demonstrable, and there IS a method paper.
# ============================================================================
import json, torch, yaml
from pathlib import Path

import sys
sys.path.insert(0, CODE_LOCAL)
from unlearning.push_scaling import (_mean_recon_error, _build,
                                     memorisation_signal, signal_to_scale,
                                     _client_train_sizes)

SEEDS = [123, 2024, 42]
N_CLIENTS = 6
HELDOUT = "val"          # matches scale_heldout_split; disjoint from both metrics
device = "cuda" if torch.cuda.is_available() else "cpu"

rows = []
for seed in SEEDS:
    OUT = f"/content/outputs_seed{seed}"
    base_ck = f"{OUT}/base_federation/round_0100/global_model.pt"
    if not Path(base_ck).exists():
        print(f"[skip] seed {seed}: no base federation at {base_ck}")
        continue

    cfg = yaml.safe_load(open(f"{CODE_LOCAL}/configs/base.yaml"))
    cfg["experiment"]["seed"] = seed
    cfg["data"]["processed_dir"] = PROCESSED_LOCAL

    teacher_params = [p.numpy() for p in
                      torch.load(base_ck, weights_only=False)["parameters"]]
    teacher = _build(teacher_params, cfg, device)
    sizes = _client_train_sizes(PROCESSED_LOCAL, N_CLIENTS, cfg)
    total = sum(sizes.values())

    # error on every client's TRAIN data (needed for the cross-client signal)
    e_train = {c: _mean_recon_error(teacher, Path(PROCESSED_LOCAL)/f"client_{c}",
                                    cfg, split="train", device=device)
               for c in range(N_CLIENTS)}

    for c in range(N_CLIENTS):
        # within-client gap (difficulty-controlled)
        e_held = _mean_recon_error(teacher, Path(PROCESSED_LOCAL)/f"client_{c}",
                                   cfg, split=HELDOUT, device=device)
        s_gap = memorisation_signal(e_train[c], e_held)

        # cross-client (sample-weighted mean over the other clients)
        num = sum(e_train[k]*sizes[k] for k in range(N_CLIENTS) if k != c)
        den = sum(sizes[k] for k in range(N_CLIENTS) if k != c)
        s_sig = memorisation_signal(e_train[c], num/den)

        rows.append({"seed": seed, "client": c,
                     "frac": sizes[c]/total,
                     "e_train": e_train[c], "e_heldout": e_held,
                     "s_gap": s_gap, "s_signal": s_sig,
                     "scale_gap": signal_to_scale(s_gap, 0.10),
                     "scale_signal": signal_to_scale(s_sig, 0.10)})
    del teacher
    if device == "cuda":
        torch.cuda.empty_cache()
    print(f"seed {seed} done")

# ------------------------------------------------------------------ report --
print("\n" + "=" * 88)
print("MEMORISATION SIGNAL ACROSS ALL CLIENTS")
print("=" * 88)
print(f"{'seed':>5}{'client':>8}{'data %':>9}{'e_train':>10}{'e_held':>10}"
      f"{'s_gap':>10}{'s_signal':>10}{'scale_gap':>11}")
for r in rows:
    print(f"{r['seed']:>5}{r['client']:>8}{100*r['frac']:>8.1f}%"
          f"{r['e_train']:>10.5f}{r['e_heldout']:>10.5f}"
          f"{r['s_gap']:>+10.4f}{r['s_signal']:>+10.4f}{r['scale_gap']:>11.3f}")

# per-client means across seeds
print("\n" + "-" * 88)
print("PER-CLIENT MEAN ACROSS SEEDS")
print(f"{'client':>8}{'data %':>9}{'mean s_gap':>13}{'mean s_signal':>15}{'mean scale':>12}")
import statistics as st
varies = False
for c in range(N_CLIENTS):
    rs = [r for r in rows if r["client"] == c]
    if not rs:
        continue
    g = st.mean(r["s_gap"] for r in rs)
    sg = st.mean(r["s_signal"] for r in rs)
    sc = st.mean(r["scale_gap"] for r in rs)
    if sc > 0.05:
        varies = True
    print(f"{c:>8}{100*rs[0]['frac']:>8.1f}%{g:>+13.4f}{sg:>+15.4f}{sc:>12.3f}")

print("\n" + "=" * 88)
if varies:
    print("VERDICT: the signal VARIES -> at least one client shows memorisation.")
    print("         Adaptivity is demonstrable. Run the ablation on those clients")
    print("         and the method paper is alive.")
else:
    print("VERDICT: the signal is ~0 for ALL clients -> constant, not adaptive.")
    print("         The 'size-aware' framing is dead. Write the validation study,")
    print("         and report this as evidence that federated AEs do not memorise")
    print("         individual clients (which corroborates the MIA finding).")
print("=" * 88)

json.dump(rows, open("/content/memorisation_signal_all_clients.json", "w"), indent=2)
import shutil
shutil.copy2("/content/memorisation_signal_all_clients.json",
             f"{OUTPUTS_DRIVE}/memorisation_signal_all_clients.json")
print("\nsaved -> memorisation_signal_all_clients.json (local + Drive)")

seed 123 done
seed 2024 done
seed 42 done

MEMORISATION SIGNAL ACROSS ALL CLIENTS
 seed  client   data %   e_train    e_held     s_gap  s_signal  scale_gap
  123       0    51.2%   0.01714   0.01773   +0.0333   -0.0517      0.333
  123       1    38.1%   0.01651   0.01779   +0.0720   +0.0209      0.720
  123       2     2.6%   0.01663   0.01524   -0.0906   +0.0064      0.000
  123       3     2.8%   0.01656   0.01659   +0.0020   +0.0107      0.020
  123       4     2.5%   0.01315   0.01327   +0.0086   +0.2182      0.086
  123       5     2.7%   0.01564   0.01757   +0.1101   +0.0672      1.000
 2024       0    51.2%   0.01636   0.01690   +0.0320   -0.0501      0.320
 2024       1    38.1%   0.01579   0.01712   +0.0777   +0.0194      0.777
 2024       2     2.6%   0.01591   0.01446   -0.1002   +0.0044      0.000
 2024       3     2.8%   0.01575   0.01582   +0.0045   +0.0148      0.045
 2024       4     2.5%   0.01255   0.01275   +0.0152   +0.2187      0.152
 2024       5     2.7%   0.014